# Binned filtered aperture E/B shear maps on HEALPix (DP2, PSZ2 G309.43-72.86, tract 2877)

Alex Broughton · October 2026. Rubin stack and Butler access required.


**Binned test.** This is a copy of `dp2_shearmap_PSZ2_G309.43-72.86.ipynb` in which the Schirmer filter is applied to
HEALPix cells of sources instead of to every galaxy. Everything up to the shear catalog is unchanged. Section 4.3
bins the sources and evaluates the binned aperture statistic, and section 8 compares it with the per-galaxy one.
Plots and output files carry the suffix `_binned`, so the per-galaxy notebook's files are not overwritten.

## Introduction

Schirmer-filtered aperture E- and B-mode shear maps on HEALPix for DP2 tract 2877, which contains the Planck SZ cluster
PSZ2 G309.43-72.86 (RA, Dec = 10.2082°, −44.1307°). By position this is SPT-CL J0040-4407 (z = 0.35,
M200 ≈ 1.4×10¹⁵ M⊙, r200 ≈ 2.0 Mpc ≈ 6.9′). Shapes are HSM regauss from the object table. These are weighted aperture
statistics, not a Kaiser-Squires reconstruction.

- E-mode: tangential shear g_t weighted by the Schirmer filter Q.
- B-mode: cross shear g× weighted by the same Q; consistent with zero for pure lensing.
- Geometry: every HEALPix cell containing a source in the target tract is evaluated at its center. Galaxies from
  neighboring tracts are included so apertures are complete at the tract edge.
- Shear frame: HSM shapes are rotated from each tract's coadd pixel grid into the local West/North (−RA, +Dec)
  frame at every source with the tract WCS. Positions use the same frame, with exact spherical distances and
  angles from astropy. g× has the standard sign, g× = −Im[g e^(−2iφ)].
- Dust: griz magnitudes are corrected for Milky Way extinction before any color cut, using the Object-table `ebv`
  column (SFD98) and the Rubin R_band coefficients (section 3.0).
- Weights: inverse variance, intrinsic shape noise plus measurement error squared.
- Filter and cells: Rs = 15′, x_c = 0.15, NSIDE = 4096 (0.86′ cells), justified in section 7.
- Binning (this notebook): sources are binned into HEALPix cells (`BIN_NSIDE`, by default the map NSIDE). Each bin
  carries its total weight, its weighted mean shear, and the shape-noise sum of its sources. Q(r) and the polar angle
  φ are evaluated once per bin, at the cell center, and the aperture sums run over bins (section 4.3).
- Peaks: S/N maxima within 0.4 Rs, aperture completeness ≥ 0.6, S/N ≥ 4, with B-mode flags (never a cut).
  They are written to a catalog in output/ and defined in peak_definition.md.

## 1.0 Setup

### 1.1 Imports and configuration

All tunable quantities live in the CONFIG dictionary.

In [ ]:
import gc
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import astropy.units as u
from astropy.coordinates import SkyCoord, angular_separation, position_angle
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.wcs import WCS
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

# The Rubin stack is optional: without it a saved local object table is read instead.
try:
    from lsst.daf.butler import Butler
    from lsst.sphgeom import Angle as SphAngle, Circle
    from lsst.geom import SpherePoint, degrees
    HAVE_LSST = True
except ImportError:
    HAVE_LSST = False
    print("Rubin stack not found: using CONFIG['LOCAL_TABLE'] instead of the Butler")

%matplotlib inline
plt.rcParams.update({"font.size": 12})


# Repo layout: _data/ holds external inputs, output/ holds generated tables/catalogs, plots/ holds figures.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "_data"
OUTPUT_DIR = ROOT / "output"
PLOTS_DIR = ROOT / "plots"
for _d in (OUTPUT_DIR, PLOTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)


# Appended to every plot and output file name, so the per-galaxy notebook's files are not overwritten.
RUN_TAG = "_binned"


def save_plot(fig, name):
    fig.savefig(PLOTS_DIR / f"{name}{RUN_TAG}.pdf", bbox_inches="tight")


def mark_cluster(ax, label=True, color="k"):
    """Mark CLUSTER on an RA/Dec axis with a star and, optionally, its name."""
    ax.scatter(CLUSTER["ra"], CLUSTER["dec"], marker="*", s=250, facecolors="none",
               edgecolors=color, linewidths=1.5, zorder=5, label=CLUSTER["name"])
    if label:
        ax.annotate(CLUSTER["name"], (CLUSTER["ra"], CLUSTER["dec"]), xytext=(8, 8),
                    textcoords="offset points", fontsize=10, color=color,
                    bbox=dict(boxstyle="round,pad=0.2", fc="w", ec="none", alpha=0.7))


def tract_area_deg2(skymap, tract):
    """Area of a tract's unique (inner) region in deg^2."""
    region = skymap[tract].inner_sky_region
    if hasattr(region, "getArea"):  # sphgeom.Box, as in rings skymaps
        sr = region.getArea()
    else:  # convex polygon: sum the solid angles of a triangle fan
        v = np.array([[p.x(), p.y(), p.z()] for p in region.getVertices()])
        a = v[0]
        sr = sum(
            2 * np.arctan2(abs(a @ np.cross(b, c)), 1 + a @ b + b @ c + c @ a)
            for b, c in zip(v[1:-1], v[2:])
        )
    return float(sr * (180.0 / np.pi) ** 2)

In [ ]:
CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",  # Butler repository root
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",  # Input collection
    "SKYMAP": "lsst_cells_v2",  # Skymap name for Butler
    "INSTRUMENT": "LSSTCam",  # Instrument dimension
    "TRACT": 2877,  # Contains PSZ2 G309.43-72.86
    # Saved copy of the tract-2877 object table, used when the Rubin stack is not importable.
    "LOCAL_TABLE": DATA_DIR / "data_PSZ2_G309.43-72.86_tract-2877.fits",
    # --- Filter, cell, and peak settings ---
    "RS_INPUT_PIX": 4500,  # Schirmer aperture radius Rs in native pixels: 4500 x 0.2" = 15'
    "PIX_SCALE_ARCSEC": 0.2,  # Arcsec per pixel (converts RS_INPUT_PIX to sky angle)
    "FILTER_XC": 0.15,  # Schirmer core x_c (NFW-optimized value; the data cannot tell 0.05-0.3 apart)
    "SEARCH_RADIUS_RS_MULT": 1.2,  # Search radius in units of Rs; Q(x) is zero for x > ~1.1
    "NSIDE": int(2**12),  # HEALPix NSIDE (RING), 0.86' cells: worst-case loss 0.7 sigma at the peak
    "PEAK_SN_MIN": 4.0,  # Peak S/N cut, above the random-rotation noise peaks; None disables
    "PEAK_EXCL_RS_MULT": 0.4,  # A peak is the S/N maximum within this many Rs (noise decorrelation length)
    "PEAK_COMPLETENESS_MIN": 0.6,  # Minimum aperture completeness (counts / expected counts); see peak_definition.md
    # B-mode information attached to each peak (never a cut).
    "PEAK_B_SN_FLAG": 3.0,  # B_FLAG when |S/N_B| >= this at the peak; also the |S/N_B| threshold for B-mode peaks
    "PEAK_B_MATCH_RS_MULT": 0.4,  # A B-mode peak within this many Rs of an E-mode peak sets B_PEAK_MATCH
    "N_RANDOM": 10,  # Random-rotation noise realizations computed alongside the map
    "RANDOM_SEED": 42,
    # --- Binning (this notebook): the Schirmer filter is applied to HEALPix cells of sources ---
    "BIN_NSIDE": None,  # HEALPix NSIDE (RING) of the source bins; None uses NSIDE (0.86' cells, 0.057 Rs)
    "BIN_POSITION": "center",  # Where Q(r) and phi are evaluated for a bin: "center" (cell center) or "centroid" (weighted mean source position)
    "BIN_WEIGHT": "wsum",  # Weight of a bin in the aperture sums: "wsum" (sum of its source weights) or "equal" (1 per occupied bin)
    # --- Shear frame and weights ---
    # Shapes and polar angles both live in the local West/North (-RA, +Dec) frame at each source (see 4.1).
    "SHEAR_FRAME": "local (-RA, +Dec)",
    "SHAPE_NOISE_E": None,  # Intrinsic per-component distortion dispersion; None measures it from the catalog
    # Source selection, following selection_cuts.md.
    "SNR_MIN": 5.0,  # Minimum per-band griz Sersic flux S/N
    "RESO_FACTOR_MIN": 0.125,  # Minimum PSF-resolution factor (star-galaxy separation)
    "COLOR_ABS_MAX": 5.0,  # Maximum |g-r|, |r-i|, |i-z| in AB magnitudes
    "EXTENDEDNESS_MIN": 0.2,  # Minimum griz model extendedness
    "E_SQ_MAX": 4.0,  # Maximum e1^2 + e2^2 for the HSM regauss distortion
    "T_MAX_ARCSEC2": None,  # Optional size cut on i_ixx + i_iyy; None disables it
    "RED_SEQ_SLOPE": 2.0484,  # Red-sequence line: g-i = slope * (r-z) + intercept
    "RED_SEQ_INTERCEPT": -0.10,  # Objects above that line are rejected (dereddened colors when DEREDDEN)
    # Milky Way dust correction, following Rubin tutorial 311.1: m_0 = m - R_band * E(B-V), with E(B-V)
    # from the Object-table ebv column (SFD98). All color cuts use the corrected magnitudes.
    "DEREDDEN": True,
    "EBV_COLUMN": "ebv",
    # R_band = A_band / E(B-V) from rubin_sim DustValues().r_x: CCM89, R_V = 3.1, flat SED, LSST baseline
    # throughputs (rubin_sim 2.6.2, throughputs_2025_08_25).
    "EXTINCTION_R": {"u": 4.757, "g": 3.661, "r": 2.701, "i": 2.054, "z": 1.590, "y": 1.308},
}

# Target cluster, with literature values for SPT-CL J0040-4407.
CLUSTER = {"name": "PSZ2 G309.43-72.86", "ra": 10.2082, "dec": -44.1307,
           "z": 0.35, "M200": 1.4e15, "c200": 4.0}

USE_BUTLER = HAVE_LSST

# Filter scale Rs expressed in the angular units needed downstream.
CONFIG["RS_ARCSEC"] = float(CONFIG["RS_INPUT_PIX"]) * float(CONFIG["PIX_SCALE_ARCSEC"])
CONFIG["RS_ARCMIN"] = CONFIG["RS_ARCSEC"] / 60.0
CONFIG["RS_DEG"] = CONFIG["RS_ARCSEC"] / 3600.0

# Aperture radius used for the neighbor search around each HEALPix cell center.
CONFIG["THETA_MAX_ARCMIN"] = float(CONFIG["SEARCH_RADIUS_RS_MULT"]) * CONFIG["RS_ARCMIN"]
CONFIG["NSIDE"] = int(CONFIG["NSIDE"])
CONFIG["BIN_NSIDE"] = int(CONFIG["NSIDE"] if CONFIG["BIN_NSIDE"] is None else CONFIG["BIN_NSIDE"])
assert CONFIG["BIN_POSITION"] in ("center", "centroid"), CONFIG["BIN_POSITION"]
assert CONFIG["BIN_WEIGHT"] in ("wsum", "equal"), CONFIG["BIN_WEIGHT"]

# Mean HEALPix pixel spacing, i.e. sqrt(pixel area), as a sanity check against Rs.
_mean_arcsec = float(np.rad2deg((4.0 * np.pi / 12.0) ** 0.5 / CONFIG["NSIDE"]) * 3600.0)
_npix = hp.nside2npix(CONFIG["NSIDE"])

print("TRACT =", CONFIG["TRACT"], "| data source:", "Butler" if USE_BUTLER else CONFIG["LOCAL_TABLE"].name)
print(f"Cluster {CLUSTER['name']}: RA={CLUSTER['ra']}, Dec={CLUSTER['dec']}")
print(
    "Schirmer Rs =",
    CONFIG["RS_INPUT_PIX"],
    "pix @",
    CONFIG["PIX_SCALE_ARCSEC"],
    "arcsec/pix →",
    f"{CONFIG['RS_ARCMIN']:.3f} arcmin, x_c = {CONFIG['FILTER_XC']}",
)
print(
    "search radius THETA_MAX_ARCMIN =",
    f"{CONFIG['THETA_MAX_ARCMIN']:.3f}",
    f"({CONFIG['SEARCH_RADIUS_RS_MULT']}×Rs) | NSIDE =",
    CONFIG["NSIDE"],
    f"(npix={_npix}; ~mean spacing {_mean_arcsec:.1f} arcsec)",
)
_bin_arcmin = float(np.rad2deg(hp.nside2resol(CONFIG["BIN_NSIDE"])) * 60.0)
print(
    f"source bins: NSIDE = {CONFIG['BIN_NSIDE']} ({_bin_arcmin:.2f}' = {_bin_arcmin / CONFIG['RS_ARCMIN']:.3f} Rs), "
    f"position = {CONFIG['BIN_POSITION']}, weight = {CONFIG['BIN_WEIGHT']}"
)

In [ ]:
# Butler client for the repo/collection defined above (skipped in local-table mode).
butler = None
if USE_BUTLER:
    butler = Butler(
        CONFIG["REPO"],
        collections=[CONFIG["COLLECTION"]],
        skymap=CONFIG["SKYMAP"],
        instrument=CONFIG["INSTRUMENT"],
    )

### 1.2 Table helpers

In [ ]:
def _col(tbl, name):
    """Column as a float64 numpy array."""
    return np.array(tbl[name]).astype(np.float64, copy=False)


def flux_to_abmag(flux_njy):
    """Convert nJy fluxes to AB magnitudes; non-positive or non-finite fluxes become NaN."""
    flux = np.asarray(flux_njy, dtype=np.float64)
    mag = np.full(flux.shape, np.nan, dtype=np.float64)
    good = np.isfinite(flux) & (flux > 0.0)
    mag[good] = (flux[good] * u.nJy).to(u.ABmag).value
    return mag


def griz_mags(tbl):
    """griz AB magnitudes from the Sersic fluxes, dust-corrected when CONFIG["DEREDDEN"] is set.

    m_0 = m - R_band * E(B-V). Objects with a non-finite E(B-V) get NaN magnitudes and fail the color cuts.
    """
    mags = {b: flux_to_abmag(_col(tbl, f"{b}_sersicFlux")) for b in "griz"}
    if not CONFIG["DEREDDEN"]:
        return mags
    ebv_col = CONFIG["EBV_COLUMN"]
    if ebv_col not in tbl.colnames:
        raise KeyError(
            f"CONFIG['DEREDDEN'] is set but the table has no '{ebv_col}' column. Read it from the Butler, "
            "re-save the local table with it, or set CONFIG['DEREDDEN'] = False."
        )
    ebv = _col(tbl, ebv_col)
    return {b: m - CONFIG["EXTINCTION_R"][b] * ebv for b, m in mags.items()}


FLAG_COLUMNS = [
    "i_i_flag",
    "i_iPSF_flag",
    "i_hsmShapeRegauss_flag",
    "i_gaapFlux_flag",
    "sersic_no_data_flag",
    "sersic_unknown_flag",
    "shape_flag",
    "i_pixelFlags_bad",
    "i_pixelFlags_nodata",
    "i_pixelFlags_edge",
    "i_pixelFlags_saturatedCenter",
    "i_pixelFlags_crCenter",
    "i_pixelFlags_interpolatedCenter",
    "i_pixelFlags_suspectCenter",
]

OBJECT_COLUMNS = [
    "objectId",
    "coord_ra",
    "coord_dec",
    "ebv",  # SFD98 E(B-V) at the object position, for the dust correction
    "detect_isDeblendedModelSource",
    "griz_model_extendedness",
    "i_hsmShapeRegauss_e1",
    "i_hsmShapeRegauss_e2",
    "i_hsmShapeRegauss_sigma",
    "i_ixx",
    "i_iyy",
    "i_ixxPSF",
    "i_iyyPSF",
    "i_inputCount",
    "g_sersicFlux",
    "g_sersicFluxErr",
    "r_sersicFlux",
    "r_sersicFluxErr",
    "i_sersicFlux",
    "i_sersicFluxErr",
    "z_sersicFlux",
    "z_sersicFluxErr",
] + FLAG_COLUMNS

SHEAR_COLUMNS = [
    "objectId",
    "coord_ra",
    "coord_dec",
    "i_hsmShapeRegauss_e1",
    "i_hsmShapeRegauss_e2",
    "i_hsmShapeRegauss_sigma",
]


def selection_mask(data):
    """Combined selection mask and the per-group masks that build it."""
    n = len(data)
    flag_columns = [c for c in FLAG_COLUMNS if c in data.colnames]

    flags_ok = np.ones(n, dtype=bool)
    for name in flag_columns:
        flags_ok &= ~np.asarray(data[name], dtype=bool)
    if "i_inputCount" in data.colnames:
        flags_ok &= np.asarray(data["i_inputCount"]) > 0

    primary_ok = np.asarray(data["detect_isDeblendedModelSource"], dtype=bool)

    e_sq = _col(data, "i_hsmShapeRegauss_e1") ** 2 + _col(data, "i_hsmShapeRegauss_e2") ** 2
    shape_ok = e_sq <= CONFIG["E_SQ_MAX"]

    t_obj = _col(data, "i_ixx") + _col(data, "i_iyy")
    with np.errstate(divide="ignore", invalid="ignore"):
        reso = 1.0 - (_col(data, "i_ixxPSF") + _col(data, "i_iyyPSF")) / t_obj
    reso_ok = reso > CONFIG["RESO_FACTOR_MIN"]

    mag = griz_mags(data)  # dust-corrected when CONFIG["DEREDDEN"]
    with np.errstate(divide="ignore", invalid="ignore"):
        snr = {b: _col(data, f"{b}_sersicFlux") / _col(data, f"{b}_sersicFluxErr") for b in "griz"}

    snr_min = CONFIG["SNR_MIN"]
    detect_ok = _col(data, "griz_model_extendedness") > CONFIG["EXTENDEDNESS_MIN"]
    for b in "griz":
        detect_ok &= snr[b] > snr_min

    color_max = CONFIG["COLOR_ABS_MAX"]
    gi = mag["g"] - mag["i"]
    rz = mag["r"] - mag["z"]
    color_ok = np.abs(mag["g"] - mag["r"]) < color_max
    color_ok &= np.abs(mag["r"] - mag["i"]) < color_max
    color_ok &= np.abs(mag["i"] - mag["z"]) < color_max
    color_ok &= np.isfinite(gi) & np.isfinite(rz)
    color_ok &= ~(gi > CONFIG["RED_SEQ_SLOPE"] * rz + CONFIG["RED_SEQ_INTERCEPT"])

    size_ok = np.ones(n, dtype=bool)
    if CONFIG["T_MAX_ARCSEC2"] is not None:
        size_ok = t_obj < CONFIG["T_MAX_ARCSEC2"] / CONFIG["PIX_SCALE_ARCSEC"] ** 2

    groups = {
        "measurement + pixel flags": flags_ok,
        "deblended model source": primary_ok,
        "|e| <= 2": shape_ok,
        f"reso_factor > {CONFIG['RESO_FACTOR_MIN']}": reso_ok,
        f"extendedness + S/N > {snr_min:g}": detect_ok,
        "colors + red sequence": color_ok,
        "size": size_ok,
    }
    mask = np.ones(n, dtype=bool)
    for cut in groups.values():
        mask &= cut
    return mask, groups


def resolve_object_columns(butler, tract):
    """Intersect OBJECT_COLUMNS with the table schema without loading the rows."""
    data_id = {"tract": int(tract), "skymap": CONFIG["SKYMAP"]}
    try:
        names = set(
            butler.get("object", dataId=data_id, storageClass="ArrowSchema").names
        )
    except Exception as exc:
        print("ArrowSchema unavailable, requesting OBJECT_COLUMNS directly:", exc)
        return list(OBJECT_COLUMNS)
    cols = [c for c in OBJECT_COLUMNS if c in names]
    missing = [c for c in OBJECT_COLUMNS if c not in names]
    print("Missing columns:", missing or "none")
    print(f"Reading {len(cols)} of {len(names)} columns")
    return cols


def load_object_table(butler, tract, columns):
    """Object table for one tract, restricted to `columns`."""
    return butler.get(
        "object",
        dataId={"tract": int(tract), "skymap": CONFIG["SKYMAP"]},
        parameters={"columns": columns},
    )


def object_tract_ids(butler):
    """Tract ids that have an object table in the configured collection."""
    data_ids = butler.registry.queryDataIds(
        ["tract"],
        datasets="object",
        collections=CONFIG["COLLECTION"],
        skymap=CONFIG["SKYMAP"],
    )
    return {int(d["tract"]) for d in data_ids}


def buffered_tract_circle(tract_info, theta_max_deg):
    """Bounding circle of the tract outer polygon, expanded by theta_max_deg."""
    bc = tract_info.getOuterSkyPolygon().getBoundingCircle()
    return Circle(
        bc.getCenter(),
        SphAngle(bc.getOpeningAngle().asRadians() + np.deg2rad(theta_max_deg)),
    )


def neighbor_tract_ids(skymap, tract_id, available, theta_max_deg):
    """Available object-table tracts whose outer polygon meets the buffered target tract."""
    buf = buffered_tract_circle(skymap[int(tract_id)], theta_max_deg)
    neighbors = []
    for tid in sorted(available):
        if tid == tract_id:
            continue
        try:
            other = skymap[int(tid)].getOuterSkyPolygon()
            if buf.intersects(other):
                neighbors.append(int(tid))
        except Exception:
            continue
    return neighbors, buf


def in_circle(ra_deg, dec_deg, circle):
    """True for points inside a sphgeom Circle."""
    ra = np.deg2rad(np.asarray(ra_deg, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec_deg, dtype=np.float64))
    cosd = np.cos(dec)
    c = circle.getCenter()
    try:
        cx, cy, cz = float(c.x()), float(c.y()), float(c.z())
    except TypeError:
        cx, cy, cz = float(c.x), float(c.y), float(c.z)
    cosang = cosd * np.cos(ra) * cx + cosd * np.sin(ra) * cy + np.sin(dec) * cz
    return cosang >= np.cos(circle.getOpeningAngle().asRadians())


def _rad(angle):
    """Plain float radians from an astropy Angle/Quantity or a bare array that is already in radians."""
    return angle.to_value(u.rad) if hasattr(angle, "to_value") else np.asarray(angle, dtype=np.float64)


def tract_wcs(tract):
    """Astropy WCS of a tract's coadd pixel grid, or None if it is not available.

    With the Butler it comes from the skymap and is cached to _data/tract-<id>_wcs.fits, so a later
    local-table run can use it too. In local-table mode only that cached file is used.
    """
    path = DATA_DIR / f"tract-{int(tract)}_wcs.fits"
    if skymap is not None:
        try:
            md = skymap[int(tract)].getWcs().getFitsMetadata()
            hdr = fits.Header()
            for key, val in md.toDict().items():
                hdr[key] = val
            try:
                fits.PrimaryHDU(header=hdr).writeto(path, overwrite=True)
            except OSError as exc:
                print(f"tract {tract}: could not cache the WCS to {path} ({exc})")
            return WCS(hdr)
        except Exception as exc:
            print(f"tract {tract}: could not convert the skymap WCS ({exc})")
    if path.exists():
        return WCS(fits.getheader(path))
    return None


def shapes_to_local_frame(e1, e2, ra, dec, wcs, step_pix=10.0):
    """Rotate pixel-frame ellipticities into the local West/North (-RA, +Dec) frame at each source.

    HSM moments are measured on the tract's coadd pixel grid. Its +x axis points West and +y North only at the
    tract center; elsewhere the axes are rotated by the convergence of meridians, and each neighbor tract has its
    own grid. The direction of each pixel axis at the source is read off the WCS with astropy position angles
    (East of North) and converted to an angle beta measured from West toward North (beta = PA + 90 deg). The
    spin-2 ellipticity then transforms as
        e_local = e_pix * exp(2i beta_x)          if the grid has the same handedness as West/North,
        e_local = conj(e_pix) * exp(2i beta_x)    if it is mirrored.
    Returns (e1_local, e2_local, beta_x in rad); with wcs=None the pixel axes are assumed to point West/North
    (exact only at the tract center) and beta_x is None.
    """
    e1 = np.asarray(e1, dtype=np.float64)
    e2 = np.asarray(e2, dtype=np.float64)
    if wcs is None:
        return e1.copy(), e2.copy(), None
    ra = np.asarray(ra, dtype=np.float64)
    dec = np.asarray(dec, dtype=np.float64)
    x, y = wcs.all_world2pix(ra, dec, 0)
    ra_x, dec_x = wcs.all_pix2world(x + step_pix, y, 0)  # one step along pixel +x
    ra_y, dec_y = wcs.all_pix2world(x, y + step_pix, 0)  # one step along pixel +y
    r = np.deg2rad
    beta_x = _rad(position_angle(r(ra), r(dec), r(ra_x), r(dec_x))) + 0.5 * np.pi
    beta_y = _rad(position_angle(r(ra), r(dec), r(ra_y), r(dec_y))) + 0.5 * np.pi
    mirrored = np.sin(beta_y - beta_x) < 0.0
    e = e1 + 1j * e2
    e = np.where(mirrored, np.conj(e), e) * np.exp(2j * beta_x)
    return e.real, e.imag, beta_x


def shear_subset(tbl, tract):
    """Keep only the columns needed after selection, plus e1_local/e2_local: the HSM shapes rotated from the
    pixel grid of `tract` into the local West/North frame at each source (see shapes_to_local_frame)."""
    e1l, e2l, beta = shapes_to_local_frame(
        _col(tbl, "i_hsmShapeRegauss_e1"), _col(tbl, "i_hsmShapeRegauss_e2"),
        _col(tbl, "coord_ra"), _col(tbl, "coord_dec"), tract_wcs(tract),
    )
    if beta is None:
        print(f"tract {tract}: no WCS available, assuming the pixel axes point West/North (exact only at the tract center)")
    elif len(tbl):
        rot = np.rad2deg(np.angle(np.exp(1j * beta)))  # pixel +x relative to West, wrapped to (-180, 180]
        print(f"tract {tract}: pixel -> local West/North rotation: median {np.median(rot):+.3f} deg, "
              f"range [{rot.min():+.3f}, {rot.max():+.3f}] deg")
    out = tbl[[c for c in SHEAR_COLUMNS if c in tbl.colnames]]
    out["e1_local"] = e1l
    out["e2_local"] = e2l
    return out


def dedupe_objectid(tbl, n_target):
    """Drop duplicate objectId rows, keeping the target-tract copy when both exist."""
    if "objectId" not in tbl.colnames:
        from_target = np.arange(len(tbl)) < n_target
        return tbl, from_target
    ids = np.asarray(tbl["objectId"])
    _, first = np.unique(ids, return_index=True)
    first.sort()
    return tbl[first], first < n_target

### 1.3 Filter, aperture, and peak functions

In [ ]:
def wrap_dra(ra, ra0):
    """RA offset ra - ra0 in degrees, wrapped to [-180, 180) so offsets stay correct across RA = 0."""
    return (np.asarray(ra, dtype=np.float64) - ra0 + 180.0) % 360.0 - 180.0


def radec_to_xyz(ra, dec):
    """Unit vectors (n, 3) for RA/Dec in degrees, for seam-free KD-tree searches."""
    ra = np.deg2rad(np.atleast_1d(np.asarray(ra, dtype=np.float64)))
    dec = np.deg2rad(np.atleast_1d(np.asarray(dec, dtype=np.float64)))
    return np.column_stack([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])


def chord_radius(theta_deg, pad=1.1):
    """Chord length on the unit sphere for an angular radius, padded so the KD-tree query is a
    superset of the exact angular cut redone inside the aperture function."""
    return 2.0 * np.sin(np.deg2rad(pad * float(theta_deg)) / 2.0)


def schirmer_weight(r, rs, xc=None):
    """Schirmer filter Q(r/Rs); r and rs in the same angular units (degrees here). xc defaults to CONFIG."""
    r = np.asarray(r, dtype=np.float64)
    rs = float(rs)
    x = r / rs

    # Schirmer (2007) coefficients: a smooth band-pass in x with a tanh core.
    a, b, c, d = 6.0, 150.0, 47.0, 50.0
    xc = CONFIG["FILTER_XC"] if xc is None else float(xc)

    # Outer envelope: rises near x ~ a/b and falls near x ~ c/d.
    q = 1.0 / (1.0 + np.exp(a - b * x) + np.exp(d * x - c))

    # Core term tanh(x/xc)/(x/xc), which tends to 1 as x -> 0.
    ratio = x / xc
    with np.errstate(divide="ignore", invalid="ignore"):
        inner = np.tanh(ratio) / ratio
    inner = np.where(np.abs(ratio) < 1e-14, 1.0, inner)

    q = q * inner
    q = np.where(np.isfinite(q), q, 0.0)  # Guard against 0/0 at the center
    return q


def ang_sep_deg(ra, dec, center):
    """Exact great-circle distance in degrees from center = (ra0, dec0), via astropy (no flat-sky approximation)."""
    ra = np.deg2rad(np.asarray(ra, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec, dtype=np.float64))
    ra0, dec0 = np.deg2rad(float(center[0])), np.deg2rad(float(center[1]))
    return np.rad2deg(_rad(angular_separation(ra0, dec0, ra, dec)))


def sep_and_phi(ra, dec, center):
    """Exact separation theta (deg) and polar angle phi (rad) of each source about center = (ra0, dec0).

    Both come from astropy's spherical trigonometry, so there is no single-cos(dec0) approximation and no
    RA = 0 seam. phi is the direction pointing away from the center, measured at the source in its local
    West/North (-RA, +Dec) frame, the same frame as the shapes (see shapes_to_local_frame). astropy position
    angles are East of North, and a direction with position angle PA has West/North polar angle PA + 90 deg.
    """
    ra = np.deg2rad(np.asarray(ra, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec, dtype=np.float64))
    ra0, dec0 = np.deg2rad(float(center[0])), np.deg2rad(float(center[1]))
    theta = np.rad2deg(_rad(angular_separation(ra0, dec0, ra, dec)))
    pa_out = _rad(position_angle(ra, dec, ra0, dec0)) + np.pi  # (source -> center) + 180 deg = outward
    return theta, pa_out + 0.5 * np.pi


def gamma_tx_sky(ra, dec, g1, g2, center, phi=None):
    """Tangential g_t and cross g_x shear about center = (ra0, dec0), in the standard convention

        g_t = -Re[g exp(-2i phi)] = -(g1 cos 2phi + g2 sin 2phi)
        g_x = -Im[g exp(-2i phi)] = -(-g1 sin 2phi + g2 cos 2phi)

    g1, g2 must be in the local West/North frame of shapes_to_local_frame, the frame phi is measured in.
    g_t does not depend on the handedness of the frame, but g_x changes sign under a mirror flip, so the
    frame is part of the convention. Pass phi (from sep_and_phi) to skip recomputing it.
    """
    g1 = np.asarray(g1, dtype=np.float64)
    g2 = np.asarray(g2, dtype=np.float64)
    if phi is None:
        _, phi = sep_and_phi(ra, dec, center)
    c2 = np.cos(2.0 * phi)
    s2 = np.sin(2.0 * phi)
    gt = -(g1 * c2 + g2 * s2)
    gx = -(-g1 * s2 + g2 * c2)
    return gt, gx


Schirmer (2007) filter Q(r/Rs) used to weight gt and g×. Beyond x ≈ c/d = 0.94 the filter is cut off sharply, so Rs is in practice the
aperture radius, and a search disc of 1.2 Rs captures all the weight. Inside the core (x < x_c = 0.15) Q is flat. Outside it Q ∝ 1/x,
the matched filter for an isothermal (SIS) shear profile g_t ∝ 1/θ. The steep term exp(6 − 150x) removes the innermost ~0.04 Rs,
where cluster members, blending, and the breakdown of the weak-shear approximation are worst.

In [ ]:
# Filter profile vs r/Rs, with Rs and the aperture search radius marked.
rs_arcmin = float(CONFIG["RS_ARCMIN"])
x_max = float(CONFIG["SEARCH_RADIUS_RS_MULT"]) * 1.15
x = np.linspace(1e-4, x_max, 800)
q = schirmer_weight(x * CONFIG["RS_DEG"], CONFIG["RS_DEG"])

fig_q, ax_q = plt.subplots(figsize=(7.0, 4.0))
ax_q.plot(x, q, "k-", lw=1.8, label=r"Q(r/Rs)")
ax_q.axvline(1.0, color="C0", ls="--", lw=1.0, label=fr"Rs = {rs_arcmin:.2f}'")
ax_q.axvline(
    CONFIG["SEARCH_RADIUS_RS_MULT"],
    color="C3",
    ls=":",
    lw=1.2,
    label=fr"search = {CONFIG['SEARCH_RADIUS_RS_MULT']}Rs",
)
ax_q.set_xlabel(r"r/Rs")
ax_q.set_ylabel(r"Q(r/Rs)")
ax_q.set_xlim(0, x_max)
ax_q.set_ylim(bottom=0)
ax_q.legend(framealpha=1)
ax_q.set_title("Schirmer aperture-mass filter")

ax_r = ax_q.twiny()
ax_r.set_xlim(0, x_max * rs_arcmin)
ax_r.set_xlabel(r"r (arcmin)")
fig_q.tight_layout()
save_plot(fig_q, "schirmer_filter")

In [ ]:
def aperture_mass_and_sn_at_center(
    ra_c,
    dec_c,
    ra_all,
    dec_all,
    g1_all,
    g2_all,
    *,
    theta_max_arcmin,
    rs_deg,
    w_all=None,
    xc=None,
    g1_rand=None,
    g2_rand=None,
):
    """Filtered aperture E- and B-mode statistics and S/N at one center; optional w_all weights galaxies.

    Returns (ap_e, sn_e, ap_b, sn_b, sn_rand). sn_rand holds the E-mode S/N for each random-rotation
    realization, from g1_rand/g2_rand (shape n_gal x K), or is None when those are not given. A
    realization rotates every galaxy by a fixed random angle (the same angle in every aperture), which
    removes the lensing signal and keeps the shape noise and source positions.
    """
    nan5 = (np.nan, np.nan, np.nan, np.nan, None)
    theta_max_deg = float(theta_max_arcmin) / 60.0
    # Exact great-circle distance and polar angle of every source (astropy): Q(r) uses the true angular r,
    # and there is no RA = 0 seam or single-cos(dec0) approximation.
    dtheta, phi_all = sep_and_phi(ra_all, dec_all, (ra_c, dec_c))

    # Restrict to sources inside the aperture; the KD-tree pre-selection is a superset of this.
    m = dtheta <= theta_max_deg
    if not np.any(m):
        return nan5

    ra_s = ra_all[m]
    dec_s = dec_all[m]
    g1_s = g1_all[m]
    g2_s = g2_all[m]
    d_theta = dtheta[m]
    phi_s = phi_all[m]
    n_g = int(ra_s.size)

    # Uniform weights unless per-object inverse variances are supplied.
    w_s = np.ones(n_g, dtype=np.float64) if w_all is None else np.asarray(w_all, dtype=np.float64)[m]

    gt, gx = gamma_tx_sky(ra_s, dec_s, g1_s, g2_s, center=(ra_c, dec_c), phi=phi_s)

    Qi = schirmer_weight(d_theta, rs_deg, xc)
    w_sum = float(np.sum(w_s))
    if w_sum <= 0.0 or not np.isfinite(w_sum):
        return nan5

    # Weighted aperture statistics: E-mode from g_t, B-mode from g_x.
    ap_e = np.sum(Qi * gt * w_s) / w_sum
    ap_b = np.sum(Qi * gx * w_s) / w_sum

    # Shape-noise error estimate, sum_i Q_i^2 |e_i|^2 w_i^2, giving S/N = sqrt(2) * signal / noise.
    denom = np.sqrt(np.sum((Qi**2) * (g1_s**2 + g2_s**2) * (w_s**2)))
    if denom <= 0.0 or not np.isfinite(denom):
        return ap_e, np.nan, ap_b, np.nan, None
    sn_e = np.sqrt(2.0) * (w_sum * ap_e) / denom
    sn_b = np.sqrt(2.0) * (w_sum * ap_b) / denom

    sn_rand = None
    if g1_rand is not None:
        # |g| is unchanged by rotation, so the noise denominator is shared by all realizations.
        c2, s2 = np.cos(2.0 * phi_s)[:, None], np.sin(2.0 * phi_s)[:, None]
        gt_r = -(g1_rand[m] * c2 + g2_rand[m] * s2)
        sn_rand = np.sqrt(2.0) * ((Qi * w_s) @ gt_r) / denom
    return ap_e, sn_e, ap_b, sn_b, sn_rand

### 1.4 Binned aperture functions

`bin_sources` bins the sources into HEALPix cells. For a cell $c$ with source weights $w_i$ it stores

$$
W_c = \sum_{i\in c} w_i, \qquad G_c = \frac{\sum_{i\in c} w_i\,g_i}{W_c}, \qquad
V_c = \frac{\sum_{i\in c} w_i^2\,|g_i|^2}{W_c^2}, \qquad n_c,
$$

the total weight, the weighted mean shear (both components, in the local West/North frame), the shape-noise term of
$G_c$, and the number of sources. `aperture_binned_at_center` then applies the filter to the bins. With bin weight
$u_c$ ($W_c$ for `BIN_WEIGHT = "wsum"`, 1 for `"equal"`), $Q_c = Q(\theta_c/R_s)$ at the bin position, and
$G_{t,c}$, $G_{\times,c}$ the tangential and cross parts of $G_c$ about the aperture center,

$$
M_{\rm ap} = \frac{\sum_c u_c\,Q_c\,G_{t,c}}{\sum_c u_c}, \qquad
\mathcal{S} = \frac{\sqrt{2}\,\sum_c u_c\,Q_c\,G_{t,c}}{\sqrt{\sum_c u_c^2\,Q_c^2\,V_c}} .
$$

With `"wsum"` this is exactly the per-galaxy estimator of 1.3 with each source's $Q$ and $\phi$ moved to its bin:
$\sum_c W_c Q_c G_{t,c} = \sum_i w_i Q_c g_{t,i}$ and $\sum_c W_c^2 Q_c^2 V_c = \sum_i Q_c^2 w_i^2 |g_i|^2$. The noise
term keeps the shape noise of every source, so the S/N is not inflated by averaging shapes. The only approximation is
the position of each source inside its cell, which matters when the cell is not small compared with the scale on
which $Q$ and $\phi$ change (the filter core, $x_c R_s = 2.25'$, and the distance to the aperture center). With
`"equal"` every occupied bin counts once, as in a pixelized map that does not track the number of sources per pixel.

The bin at the aperture center itself ($\theta = 0$, where $\phi$ is undefined) is skipped. $Q(0) \approx 0.002$, so it
carries almost no weight. Random rotations are applied per galaxy before binning, so $V_c$ is shared by all
realizations.

In [ ]:
def bin_sources(nside_bin, ra, dec, g1_, g2_, w_, g1_rand=None, g2_rand=None, position="center"):
    """Bin sources into HEALPix RING cells of nside_bin (see 1.4).

    Returns a dict with ipix, ra, dec (where Q and phi are evaluated), W, G1, G2, V, n and, when g1_rand/g2_rand
    (n_gal x K) are given, the binned random shears G1r, G2r (n_bin x K). Shapes are averaged in each source's own
    local West/North frame; across a 0.86' cell that frame turns by ~0.01 deg, which is negligible.
    """
    ra = np.asarray(ra, dtype=np.float64)
    dec = np.asarray(dec, dtype=np.float64)
    w_ = np.asarray(w_, dtype=np.float64)
    ipix = hp.ang2pix(int(nside_bin), ra, dec, nest=False, lonlat=True)
    cells, inv, n_c = np.unique(ipix, return_inverse=True, return_counts=True)
    nb = int(cells.size)

    def _sum(x):
        return np.bincount(inv, weights=x, minlength=nb)

    W = _sum(w_)
    ok = W > 0.0
    W_safe = np.where(ok, W, 1.0)
    G1 = np.where(ok, _sum(w_ * g1_) / W_safe, 0.0)
    G2 = np.where(ok, _sum(w_ * g2_) / W_safe, 0.0)
    V = np.where(ok, _sum(w_**2 * (g1_**2 + g2_**2)) / W_safe**2, 0.0)

    ra_b, dec_b = hp.pix2ang(int(nside_bin), cells, nest=False, lonlat=True)
    ra_b, dec_b = np.asarray(ra_b, dtype=np.float64), np.asarray(dec_b, dtype=np.float64)
    if position == "centroid":
        # Weighted mean unit vector of the sources (no RA = 0 seam); cells with no weight keep their center.
        xyz = radec_to_xyz(ra, dec) * w_[:, None]
        v = np.column_stack([_sum(xyz[:, k]) for k in range(3)])
        norm = np.linalg.norm(v, axis=1)
        has = ok & (norm > 0.0)
        v = v[has] / norm[has, None]
        ra_b[has] = np.rad2deg(np.arctan2(v[:, 1], v[:, 0])) % 360.0
        dec_b[has] = np.rad2deg(np.arcsin(np.clip(v[:, 2], -1.0, 1.0)))
    elif position != "center":
        raise ValueError(f"unknown bin position {position!r}")

    out = dict(nside=int(nside_bin), ipix=cells, ra=ra_b, dec=dec_b, W=W, G1=G1, G2=G2, V=V, n=n_c)
    if g1_rand is not None:
        K = g1_rand.shape[1]
        out["G1r"] = np.empty((nb, K))
        out["G2r"] = np.empty((nb, K))
        for j in range(K):
            out["G1r"][:, j] = np.where(ok, _sum(w_ * g1_rand[:, j]) / W_safe, 0.0)
            out["G2r"][:, j] = np.where(ok, _sum(w_ * g2_rand[:, j]) / W_safe, 0.0)
    return out


def bin_tree(bins):
    """KD-tree on the unit vectors of the bin positions (no RA = 0 seam)."""
    return cKDTree(radec_to_xyz(bins["ra"], bins["dec"]))


def aperture_binned_at_center(ra_c, dec_c, bins, idx, *, theta_max_arcmin, rs_deg, xc=None, weight=None, rand=False):
    """Binned E- and B-mode aperture statistics and S/N at one center (see 1.4).

    idx are candidate bin indices (a superset of the bins within theta_max, e.g. from bin_tree). Returns
    (ap_e, sn_e, ap_b, sn_b, sn_rand, count): sn_rand is the E-mode S/N of each random realization (rand=True and
    the bins carry G1r/G2r) or None, and count is the number of sources in bins within theta_max.
    """
    weight = CONFIG["BIN_WEIGHT"] if weight is None else weight
    idx = np.asarray(idx, dtype=np.int64)
    if idx.size == 0:
        return np.nan, np.nan, np.nan, np.nan, None, 0
    theta_max_deg = float(theta_max_arcmin) / 60.0
    theta, phi = sep_and_phi(bins["ra"][idx], bins["dec"][idx], (ra_c, dec_c))
    inside = theta <= theta_max_deg
    count = int(np.sum(bins["n"][idx][inside]))
    m = inside & (theta > 1e-9)  # skip the bin at the center, where phi is undefined
    if not np.any(m):
        return np.nan, np.nan, np.nan, np.nan, None, count
    sel = idx[m]
    if weight == "wsum":
        u_c = bins["W"][sel]
    elif weight == "equal":
        u_c = (bins["W"][sel] > 0.0).astype(np.float64)
    else:
        raise ValueError(f"unknown bin weight {weight!r}")
    u_sum = float(np.sum(u_c))
    if u_sum <= 0.0 or not np.isfinite(u_sum):
        return np.nan, np.nan, np.nan, np.nan, None, count

    Qc = schirmer_weight(theta[m], rs_deg, xc)
    gt, gx = gamma_tx_sky(None, None, bins["G1"][sel], bins["G2"][sel], (ra_c, dec_c), phi=phi[m])
    uq = u_c * Qc
    ap_e = np.sum(uq * gt) / u_sum
    ap_b = np.sum(uq * gx) / u_sum

    denom = np.sqrt(np.sum(uq**2 * bins["V"][sel]))
    if denom <= 0.0 or not np.isfinite(denom):
        return ap_e, np.nan, ap_b, np.nan, None, count
    sn_e = np.sqrt(2.0) * np.sum(uq * gt) / denom
    sn_b = np.sqrt(2.0) * np.sum(uq * gx) / denom

    sn_rand = None
    if rand and "G1r" in bins:
        c2, s2 = np.cos(2.0 * phi[m])[:, None], np.sin(2.0 * phi[m])[:, None]
        gt_r = -(bins["G1r"][sel] * c2 + bins["G2r"][sel] * s2)
        sn_rand = np.sqrt(2.0) * (uq @ gt_r) / denom
    return ap_e, sn_e, ap_b, sn_b, sn_rand, count

In [ ]:
def find_peaks_shear_healpix(
    ipix_eval,
    sn_map,
    completeness,
    nside,
    excl_radius_arcmin,
    sn_min=None,
    completeness_min=0.6,
):
    """Peaks of an S/N map on HEALPix cells.

    A cell is a peak if (1) its S/N is the highest among all evaluated cells within excl_radius_arcmin,
    (2) its aperture completeness is at least completeness_min, and (3) its S/N is at least sn_min.
    Using S/N rather than M_ap means a noisy shallow region does not produce large-|M_ap| peaks. An
    exclusion radius of about the noise decorrelation length (~0.4 Rs) stops one noise bump from
    being counted several times. Returns the HEALPix indices of the peaks, highest S/N first.
    """
    ipix_eval = np.asarray(ipix_eval, dtype=np.int64)
    sn_map = np.asarray(sn_map, dtype=np.float64)
    completeness = np.asarray(completeness, dtype=np.float64)

    # ipix_eval is sorted (np.unique), so HEALPix index -> array position is a searchsorted lookup.
    srt = np.argsort(ipix_eval)
    ip_sorted = ipix_eval[srt]

    cand = np.isfinite(sn_map) & (completeness >= completeness_min)
    if sn_min is not None:
        cand &= sn_map >= sn_min
    order = np.flatnonzero(cand)
    order = order[np.argsort(-sn_map[order])]

    excl_rad = np.deg2rad(float(excl_radius_arcmin) / 60.0)
    peaks = []
    for k in order:
        vec = hp.pix2vec(nside, int(ipix_eval[k]), nest=False)
        disc = hp.query_disc(nside, vec, excl_rad, inclusive=False, nest=False)
        pos = np.clip(np.searchsorted(ip_sorted, disc), 0, ip_sorted.size - 1)
        j = srt[pos[ip_sorted[pos] == disc]]
        vals = sn_map[j]
        if np.all(~np.isfinite(vals) | (vals <= sn_map[k]) | (j == k)):
            peaks.append(int(ipix_eval[k]))
    return np.asarray(peaks, dtype=np.int64)


def match_bmode_peaks(
    peak_ipix,
    ipix_eval,
    sn_b_map,
    completeness,
    nside,
    excl_radius_arcmin,
    b_sn_min,
    match_radius_arcmin,
    completeness_min=0.6,
):
    """Nearest B-mode peak to each E-mode peak. Informational only: no E-mode peak is removed.

    B-mode peaks are found with find_peaks_shear_healpix on +S/N_B and on -S/N_B (a systematic can have either
    sign), with the same exclusion radius and completeness cut as the E-mode peaks and |S/N_B| >= b_sn_min.
    Returns a dict with, per E-mode peak, B_PEAK_SN (signed), B_PEAK_RA, B_PEAK_DEC, B_PEAK_SEP_ARCMIN and
    B_PEAK_MATCH (separation <= match_radius_arcmin), plus the B-mode peaks themselves (b_ipix, b_sn).
    The separation is kept beyond the match radius, so the match can be redone later with another radius.
    """
    peak_ipix = np.asarray(peak_ipix, dtype=np.int64)
    sn_b_map = np.asarray(sn_b_map, dtype=np.float64)
    kw = dict(nside=nside, excl_radius_arcmin=excl_radius_arcmin, sn_min=b_sn_min, completeness_min=completeness_min)
    b_ipix = np.concatenate([
        find_peaks_shear_healpix(ipix_eval, sn_b_map, completeness, **kw),
        find_peaks_shear_healpix(ipix_eval, -sn_b_map, completeness, **kw),
    ])
    idx_of = {int(p): k for k, p in enumerate(ipix_eval)}
    b_sn = np.array([sn_b_map[idx_of[int(p)]] for p in b_ipix], dtype=np.float64)

    n = peak_ipix.size
    out = {key: np.full(n, np.nan) for key in ("B_PEAK_SN", "B_PEAK_RA", "B_PEAK_DEC", "B_PEAK_SEP_ARCMIN")}
    out["B_PEAK_MATCH"] = np.zeros(n, dtype=bool)
    if n and b_ipix.size:
        ra_b, dec_b = hp.pix2ang(nside, b_ipix, nest=False, lonlat=True)
        ra_e, dec_e = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
        for i in range(n):
            sep = ang_sep_deg(ra_b, dec_b, (ra_e[i], dec_e[i])) * 60.0
            j = int(np.argmin(sep))
            out["B_PEAK_SN"][i], out["B_PEAK_RA"][i], out["B_PEAK_DEC"][i] = b_sn[j], ra_b[j], dec_b[j]
            out["B_PEAK_SEP_ARCMIN"][i] = sep[j]
            out["B_PEAK_MATCH"][i] = sep[j] <= float(match_radius_arcmin)
    out["b_ipix"], out["b_sn"] = b_ipix, b_sn
    return out

## 2.0 Load data

In [ ]:
theta_max_deg = CONFIG["THETA_MAX_ARCMIN"] / 60.0
NEIGHBOR_TRACTS, TRACT_BUFFER_CIRCLE = [], None

if USE_BUTLER:
    object_columns = resolve_object_columns(butler, CONFIG["TRACT"])
    available_tracts = object_tract_ids(butler)
    skymap = butler.get("skyMap", dataId={"skymap": CONFIG["SKYMAP"]})

    # The cluster must lie in the target tract for the map to cover it.
    _cluster_tract = skymap.findTract(SpherePoint(CLUSTER["ra"], CLUSTER["dec"], degrees)).getId()
    print(f"{CLUSTER['name']} lies in tract {_cluster_tract}")
    assert _cluster_tract == CONFIG["TRACT"], "set CONFIG['TRACT'] to the cluster tract"

    try:
        NEIGHBOR_TRACTS, TRACT_BUFFER_CIRCLE = neighbor_tract_ids(
            skymap, CONFIG["TRACT"], available_tracts, theta_max_deg
        )
    except Exception as exc:
        print("Neighbor lookup failed, mapping the target tract only:", exc)

    data = load_object_table(butler, CONFIG["TRACT"], object_columns)
else:
    skymap = None
    data = Table.read(CONFIG["LOCAL_TABLE"])
    object_columns = [c for c in OBJECT_COLUMNS if c in data.colnames]

print(f"Target tract {CONFIG['TRACT']}: {len(data):,} rows")
print("Neighbor tracts:", NEIGHBOR_TRACTS or "none")

## 3.0 Source selection

The cuts follow selection_cuts.md, which is written for the metadetect catalog
(object_shear_all). Here we use the object table and HSM regauss shapes, so each cut
is applied through its object-table counterpart:

| selection_cuts.md (metadetect) | Used here (object table) |
| --- | --- |
| all *_flags false | i-band measurement, PSF, shape, GAaP, and Sersic flags |
| bmask_flags, image_flags, mfrac < 0.1 | i-band pixelFlags_* (bad, nodata, edge, saturatedCenter, crCenter, interpolatedCenter, suspectCenter) |
| is_primary | detect_isDeblendedModelSource |
| SNR > 10 | griz Sersic flux S/N > SNR_MIN (set in CONFIG) |
| reso_factor > 0.5 | 1 - T_PSF/T_obj from the i-band second moments, > RESO_FACTOR_MIN (set in CONFIG) |
| \|g-r\|, \|r-i\|, \|i-z\| < 5 | same, on AB magnitudes from the Sersic fluxes |
| gauss_T < 20 | optional cut on i_ixx + i_iyy, disabled by default |

There is no per-object masked-pixel fraction in the object table, so mfrac is covered only
indirectly by the pixel flags. Two cuts specific to this analysis are added on top: a bound on
the HSM distortion, and a red-sequence veto.

All color cuts use magnitudes corrected for Milky Way dust, $m_0 = m - R_{\rm band}\,E(B-V)$, following Rubin
tutorial 311.1. E(B−V) is the Object-table `ebv` column (SFD98 at each object's position) and R_band comes from
rubin_sim `DustValues().r_x` (CCM89, R_V = 3.1, flat SED): R_g, R_r, R_i, R_z = 3.661, 2.701, 2.054, 1.590.
Without the correction, reddening moves red-sequence galaxies below the veto line by about
(R_g − R_i) − 2.0484 (R_r − R_z) ≈ −0.67 E(B−V) mag in g−i, so they leak into the source sample in dusty tracts.
Set `CONFIG["DEREDDEN"] = False` to turn the correction off.

### 3.1 Per-object quantities

In [ ]:
# Sersic fluxes and their errors, in nJy.
g_flux, g_err = _col(data, "g_sersicFlux"), _col(data, "g_sersicFluxErr")
r_flux, r_err = _col(data, "r_sersicFlux"), _col(data, "r_sersicFluxErr")
i_flux, i_err = _col(data, "i_sersicFlux"), _col(data, "i_sersicFluxErr")
z_flux, z_err = _col(data, "z_sersicFlux"), _col(data, "z_sersicFluxErr")

# AB magnitudes, dust-corrected when CONFIG["DEREDDEN"]; non-positive fluxes become NaN and are dropped by the
# color cuts below.
_mags = griz_mags(data)
g_mag, r_mag, i_mag, z_mag = (_mags[b] for b in "griz")
if CONFIG["DEREDDEN"]:
    _ebv = _col(data, CONFIG["EBV_COLUMN"])
    print(
        f"E(B-V) in tract {CONFIG['TRACT']}: median {np.nanmedian(_ebv):.3f}, "
        f"range {np.nanmin(_ebv):.3f} to {np.nanmax(_ebv):.3f}"
    )
    print("A_band at the median:", {b: round(CONFIG['EXTINCTION_R'][b] * float(np.nanmedian(_ebv)), 3) for b in 'griz'})

# Adjacent-band colors for the sanity cuts, plus (g-i, r-z) for the red-sequence line.
gr = g_mag - r_mag
ri = r_mag - i_mag
iz = i_mag - z_mag
gi = g_mag - i_mag
rz = r_mag - z_mag

# Per-band detection significance.
g_snr = g_flux / g_err
r_snr = r_flux / r_err
i_snr = i_flux / i_err
z_snr = z_flux / z_err

# Resolution factor: 1 - (PSF second moments) / (object second moments).
# Values near 0 are PSF-like, so stars sit near the bottom of the diagnostic below.
reso_factor = 1 - (_col(data, "i_ixxPSF") + _col(data, "i_iyyPSF")) / (
    _col(data, "i_ixx") + _col(data, "i_iyy")
)

### 3.2 Diagnostics

Color-color and star-galaxy separation planes, with the cut thresholds drawn on top.

In [ ]:
# Color-color plane before any cuts, with the red-sequence veto line from CONFIG.
_cc_tag = ", dereddened" if CONFIG["DEREDDEN"] else ""

def cc_seq_parallel(rz):
    return CONFIG["RED_SEQ_SLOPE"] * rz + CONFIG["RED_SEQ_INTERCEPT"]

plt.hexbin(rz, gi, mincnt=1, extent=(-1,3,-1,3))
s = np.linspace(-5,5, 1000)
plt.plot(s, cc_seq_parallel(s), "r-")
plt.ylabel(f"g - i (AB mag{_cc_tag})")
plt.xlabel(f"r - z (AB mag{_cc_tag})")
plt.colorbar()
plt.xlim(-1,3)
plt.ylim(-1,3)

In [ ]:
# Star-galaxy separation plane; the dashed lines mark the cuts applied in 3.3.
plt.plot(i_snr, reso_factor, ',')
plt.axhline(CONFIG["RESO_FACTOR_MIN"], ls='--', color='k', alpha=0.5)
plt.axvline(CONFIG["SNR_MIN"], ls='--', color='k', alpha=0.5)
plt.xlabel("i-band Sersic flux S/N")
plt.ylabel("resolution factor")
plt.ylim(-0.5, 1.0)
plt.xlim(1, 1000)
plt.semilogx()
save_plot(plt.gcf(), f"{CONFIG['TRACT']}_star_galaxy")

### 3.3 Apply the cuts

The red-sequence veto removes galaxies above a line of slope 2.0484 (intercept −0.10) in the dereddened (r−z, g−i) plane. These are likely
cluster members rather than lensed background sources, so they would dilute the signal.

In [ ]:
mask, groups = selection_mask(data)
shear_catalog = data[mask]
from_target = np.ones(len(shear_catalog), dtype=bool)

for _name, _cut in groups.items():
    print(f"{_name:<28} {int(np.sum(_cut)):>10,}")
print(f"{'all cuts':<28} {len(shear_catalog):>10,} of {len(data):,}")

In [ ]:
# Source density over the tract's unique (inner) area. Without the skymap, the area is the
# footprint of occupied NSIDE=1024 (3.4') cells, which slightly overestimates the area at the edges.
if skymap is not None:
    tract_area = tract_area_deg2(skymap, CONFIG["TRACT"])
else:
    _ip = hp.ang2pix(1024, _col(shear_catalog, "coord_ra"), _col(shear_catalog, "coord_dec"), lonlat=True)
    tract_area = np.unique(_ip).size * hp.nside2pixarea(1024, degrees=True)
print(f"tract area: {tract_area:.3f} deg^2")
N_DENSITY = len(shear_catalog) / (tract_area * 3600.0)  # gal / arcmin^2, used for aperture completeness
print(round(N_DENSITY, 3), "gal / arcmin^2")

In [ ]:
# Same color-color plane after the cuts: the red sequence should be removed.
_mags_sel = griz_mags(shear_catalog)  # same (dust-corrected) magnitudes as the cuts
plt.hexbin(
    _mags_sel["r"] - _mags_sel["z"],
    _mags_sel["g"] - _mags_sel["i"], mincnt=1, extent=(-1,3,-1,3))
s = np.linspace(-5,5, 1000)
plt.plot(s, cc_seq_parallel(s), "r-")
plt.ylabel(f"g - i (AB mag{_cc_tag})")
plt.xlabel(f"r - z (AB mag{_cc_tag})")
plt.colorbar()
plt.xlim(-1,3)
plt.ylim(-1,3)

Number of selected galaxies per HEALPix cell after the cuts.

In [ ]:
# Occupied HEALPix cells after the selection cuts (target tract only).
nside_gal = int(CONFIG["NSIDE"])
ra_sel = _col(shear_catalog, "coord_ra")
dec_sel = _col(shear_catalog, "coord_dec")
ipix_sel = hp.ang2pix(nside_gal, ra_sel, dec_sel, nest=False, lonlat=True)
ipix_occ, n_gal_cell = np.unique(ipix_sel, return_counts=True)
ra_cell, dec_cell = hp.pix2ang(nside_gal, ipix_occ, nest=False, lonlat=True)

print(
    f"Occupied cells: {ipix_occ.size:,}  |  "
    f"galaxies/cell: min={int(n_gal_cell.min())}  "
    f"median={np.median(n_gal_cell):.0f}  "
    f"mean={n_gal_cell.mean():.1f}  "
    f"max={int(n_gal_cell.max())}"
)

fig_ngal, axes_ngal = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
pt_ngal = max(1.0, min(8.0, 500_000.0 / max(int(ipix_occ.size), 1)))
sc_ngal = axes_ngal[0].scatter(
    ra_cell,
    dec_cell,
    c=n_gal_cell,
    s=pt_ngal,
    cmap="magma",
    linewidths=0,
    alpha=0.85,
)
axes_ngal[0].set_xlabel("RA (deg)")
axes_ngal[0].set_ylabel("Dec (deg)")
axes_ngal[0].set_title(f"Galaxies per HEALPix cell (NSIDE={nside_gal})")
axes_ngal[0].invert_xaxis()
fig_ngal.colorbar(sc_ngal, ax=axes_ngal[0], fraction=0.046, pad=0.02, label="N galaxies")

n_max = int(n_gal_cell.max())
bins_ngal = np.arange(0.5, n_max + 1.5, 1) if n_max < 80 else 40
axes_ngal[1].hist(n_gal_cell, bins=bins_ngal, histtype="stepfilled", color="0.35")
axes_ngal[1].set_xlabel("Galaxies per cell")
axes_ngal[1].set_ylabel("Number of cells")
axes_ngal[1].set_title("Occupancy after selection cuts")
save_plot(fig_ngal, f"{CONFIG['TRACT']}_ngal_cells")

### 3.4 Neighbor-tract buffer

Selected galaxies from neighboring tracts within the search radius of the target tract.

In [ ]:
n_target = len(shear_catalog)
pieces = [shear_subset(shear_catalog, CONFIG["TRACT"])]
del shear_catalog, data
gc.collect()

for tid in NEIGHBOR_TRACTS:  # empty in local-table mode
    try:
        tbl = load_object_table(butler, tid, object_columns)
    except Exception as exc:
        print(f"tract {tid}: skipped ({exc})")
        continue
    nb_mask, _ = selection_mask(tbl)
    sel = shear_subset(tbl[nb_mask], tid)
    del tbl
    if len(sel) == 0:
        gc.collect()
        continue
    keep = in_circle(sel["coord_ra"], sel["coord_dec"], TRACT_BUFFER_CIRCLE)
    n_keep = int(np.sum(keep))
    if n_keep:
        pieces.append(sel[keep])
    del sel
    gc.collect()
    print(f"tract {tid}: {n_keep:,} galaxies in buffer")

shear_catalog = vstack(pieces, join_type="inner") if len(pieces) > 1 else pieces[0]
del pieces
gc.collect()

shear_catalog, from_target = dedupe_objectid(shear_catalog, n_target)
print(
    f"Shear catalog: {int(from_target.sum()):,} in tract, "
    f"{int((~from_target).sum()):,} from neighbors "
    f"({len(shear_catalog):,} unique)"
)

## 4.0 Map on HEALPix

### 4.1 Shear calibration, frame, and weights

Regauss distortions are converted to shear with the responsivity R and multiplicative bias
m calibrated on Abell 360: $g = e / [2R(1+m)]$.

**Shear frame.** `i_hsmShapeRegauss_e1/e2` are measured from pixel moments on each tract's coadd, so they are
defined in that tract's pixel frame. Its +x axis points toward −RA (West) and +y toward +Dec (North), but exactly so
only at the tract center. Elsewhere the axes are rotated by the convergence of the meridians (by about
0.85°·|tan δ| at the tract edge, ≈ 0.8° at δ = −44°), and each neighbor tract has its own pixel grid. A rotation by α
leaks a fraction ≈ 2α of $g_t$ into $g_\times$. In 3.4 every shape is therefore rotated into the local West/North
frame at its own position (`shapes_to_local_frame`). The pixel-axis directions are read off the tract WCS with astropy
position angles, and the spin-2 ellipticity is rotated by twice that angle (and conjugated if the grid were mirrored).
The polar angle φ of each source about an aperture center is measured in the same local West/North frame
(`sep_and_phi`: astropy `position_angle`, which is East of North, converted with φ = PA + 90°). Distances are exact
great-circle separations (astropy `angular_separation`), so there is no single-cos δ₀ flat-sky approximation and no
RA = 0 seam. Shapes and positions must share a frame. With a mirror flip between them, a purely tangential field
($g_1 = -g_t\cos2\phi$, $g_2 = -g_t\sin2\phi$) is measured as $g_t\cos4\phi$ in the E-mode and $g_t\sin4\phi$ in the
B-mode, and the lensing signal averages to zero in both maps.

If the tract WCS is not available (local-table mode without a cached `_data/tract-<id>_wcs.fits`, which a Butler run
writes), the pixel axes are assumed to point exactly West/North, and the run prints a warning.

**Sign convention.** $g_t = -{\rm Re}[g\,e^{-2i\phi}] = -(g_1\cos2\phi + g_2\sin2\phi)$ and
$g_\times = -{\rm Im}[g\,e^{-2i\phi}] = g_1\sin2\phi - g_2\cos2\phi$. $g_t$ does not depend on the handedness of the
frame, but $g_\times$ changes sign under a mirror flip. The West/North frame has the handedness of the coadd pixel
grid, so $g_\times$ here matches pixel-coordinate pipelines (A360, LoVoCCS). To compare with a catalog whose shapes are
defined with x toward +RA (East), flip the sign of its $e_2$ (or of its $g_\times$) first.

**Weights.** $w = 1/(\sigma_{e,\rm int}^2 + \sigma_e^2)$, where $\sigma_e$ is `i_hsmShapeRegauss_sigma`, the per-component
distortion error. The intrinsic dispersion is estimated from the catalog as
$\sigma_{e,\rm int}^2 = \langle e_1^2+e_2^2\rangle/2 - {\rm median}(\sigma_e^2)$.

In [ ]:
# Quick check that the mean ellipticity is small; a large value signals additive bias.
np.mean(shear_catalog['i_hsmShapeRegauss_e1'])

In [ ]:
# Responsivity and multiplicative bias from the A360 calibration.
R, m_mean = 0.8511839657185559, -0.13584282915829576

ra_deg = _col(shear_catalog, "coord_ra")
dec_deg = _col(shear_catalog, "coord_dec")

# Distortion -> shear: g = e / (2R) / (1 + m), from the shapes already rotated into each source's
# local West/North frame (shear_subset in 3.4), the same frame the polar angles are measured in.
e1_0 = _col(shear_catalog, "e1_local")
e2_0 = _col(shear_catalog, "e2_local")
g1 = (e1_0 / (2.0 * R)) / (1.0 + m_mean)
g2 = (e2_0 / (2.0 * R)) / (1.0 + m_mean)

# Weights: 1 / (intrinsic distortion variance + measurement variance), both per component.
sig_e = _col(shear_catalog, "i_hsmShapeRegauss_sigma")
if CONFIG["SHAPE_NOISE_E"] is None:
    _s2 = np.nanmean(e1_0**2 + e2_0**2) / 2.0 - np.nanmedian(sig_e**2)
    CONFIG["SHAPE_NOISE_E"] = float(np.sqrt(max(_s2, 0.05)))
_var_sum = CONFIG["SHAPE_NOISE_E"] ** 2 + sig_e**2
with np.errstate(divide="ignore", invalid="ignore"):
    w_gal = np.where(np.isfinite(_var_sum) & (_var_sum > 0.0), 1.0 / _var_sum, 0.0)

# Effective source density and per-component shear dispersion.
N_EFF = N_DENSITY * np.sum(w_gal[from_target]) ** 2 / np.sum(w_gal[from_target] ** 2) / from_target.sum()
SIGMA_G = float(np.sqrt(np.sum(w_gal**2 * (g1**2 + g2**2) / 2) / np.sum(w_gal**2)))

print("N galaxies:", ra_deg.size)
print("Galaxies with positive finite weight:", int(np.sum(w_gal > 0)))
print(f"sigma_e,int (distortion, per component) = {CONFIG['SHAPE_NOISE_E']:.3f}")
print(f"n_eff = {N_EFF:.1f} / arcmin^2,  sigma_g (shear, per component, weighted) = {SIGMA_G:.3f}")

In [ ]:
# Effective source density from the weights above (target-tract sources only).
_t = from_target & (w_gal > 0)
_w = w_gal[_t]
_area = tract_area * 3600.0  # arcmin^2
sig_int = CONFIG["SHAPE_NOISE_E"] / (2.0 * R * (1.0 + m_mean))  # intrinsic shape noise, shear per component

# Heymans+12: n_eff = (sum w)^2 / (A sum w^2), paired with the weighted shear dispersion.
n_eff_H = np.sum(_w) ** 2 / np.sum(_w**2) / _area
sig_H = np.sqrt(np.sum(_w**2 * (g1[_t] ** 2 + g2[_t] ** 2) / 2) / np.sum(_w**2))

# Chang+13: n_eff = sum_i sigma_int^2 / (sigma_int^2 + sigma_i^2) / A, paired with sigma_int.
n_eff_C = CONFIG["SHAPE_NOISE_E"] ** 2 * np.sum(_w) / _area

print(f"raw n = {_t.sum() / _area:.2f} / arcmin^2")
print(f"Heymans+12: n_eff = {n_eff_H:.2f} / arcmin^2, sigma = {sig_H:.3f}, sigma^2/n_eff = {sig_H**2 / n_eff_H:.2e} arcmin^2")
print(f"Chang+13:   n_eff = {n_eff_C:.2f} / arcmin^2, sigma = {sig_int:.3f}, sigma^2/n_eff = {sig_int**2 / n_eff_C:.2e} arcmin^2")
print(f"equivalent n at sigma = 0.25 (Frieman 2026): {0.25**2 * n_eff_H / sig_H**2:.2f} / arcmin^2")

### 4.2 Tangential shear profile around the cluster

Weighted mean $g_t$ and $g_\times$ in log-spaced annuli around the SZ position.

In [ ]:
def tangential_profile(g1_, g2_, w_, center, bins_arcmin):
    """Weighted <g_t>, <g_x> and shape-noise errors in annuli about center (ra, dec)."""
    r, phi_ = sep_and_phi(ra_deg, dec_deg, center)  # exact separation (deg) and polar angle
    r = r * 60.0  # arcmin
    gt_, gx_ = gamma_tx_sky(ra_deg, dec_deg, g1_, g2_, center, phi=phi_)
    rc, mt, mx, err = [], [], [], []
    for lo, hi in zip(bins_arcmin[:-1], bins_arcmin[1:]):
        s = (r >= lo) & (r < hi)
        ws = w_[s]
        rc.append(np.sqrt(lo * hi))
        mt.append(np.sum(ws * gt_[s]) / ws.sum())
        mx.append(np.sum(ws * gx_[s]) / ws.sum())
        err.append(np.sqrt(np.sum(ws**2 * (g1_[s] ** 2 + g2_[s] ** 2) / 2)) / ws.sum())
    return np.array(rc), np.array(mt), np.array(mx), np.array(err)


PROFILE_BINS = np.geomspace(0.75, 25.0, 12)
prof = tangential_profile(g1, g2, w_gal, (CLUSTER["ra"], CLUSTER["dec"]), PROFILE_BINS)
rc, mt, mx, err = prof

fig_p, ax_p = plt.subplots(figsize=(7, 4.5))
ax_p.errorbar(rc, mt, err, fmt="o", color="C3", label=r"$g_t$")
ax_p.errorbar(rc * 1.05, mx, err, fmt="s", mfc="none", color="0.4", label=r"$g_\times$")
ax_p.axhline(0, color="k", lw=0.8)
ax_p.set_xscale("log")
ax_p.set_xlabel(f"θ from {CLUSTER['name']} (arcmin)")
ax_p.set_ylabel("shear")
ax_p.set_title("Tangential and cross shear profile")
ax_p.legend()
fig_p.tight_layout()
save_plot(fig_p, f"{CONFIG['TRACT']}_PSZ2_G309_tangential_profile")

### 4.3 Binned aperture statistic on every cell

The sources (target tract plus neighbor buffer) are binned into HEALPix cells of `BIN_NSIDE` (section 1.4), and the
filter is applied to the bins. The evaluation grid is unchanged: every occupied NSIDE cell of the target tract is
evaluated at its center. With `BIN_NSIDE = NSIDE` the bins and the evaluation cells are the same cells.

`N_RANDOM` noise realizations are computed alongside the map. In each, every galaxy is rotated by a random angle that
stays fixed across apertures (the same angles as in the per-galaxy notebook for the same seed), and the rotated shapes
are then binned. These maps keep the real positions, weights, masks, and depth variations but contain no lensing
signal.

Aperture completeness is the number of sources in bins within the search disc divided by `N_DENSITY · π θ_max²`. It is
low near tract edges when neighbors are missing, and around large masks.

In [ ]:
def run_aperture_map(nside, rs_deg, theta_max_arcmin, xc=None, n_random=0, eval_mask=None, seed=0, progress=True,
                     bin_nside=None):
    """Evaluate the binned aperture statistic on every occupied HEALPix cell of the target tract.

    The sources are binned at bin_nside (default CONFIG["BIN_NSIDE"]) and the filter is applied to the bins
    (bin_sources, aperture_binned_at_center). Returns a dict with ipix, ap_e, sn_e, ap_b, sn_b, counts,
    completeness, sn_rand (n_cell x n_random), the bins, and the run time in seconds.
    eval_mask optionally restricts the evaluated cells (boolean over the occupied cells).
    """
    t0 = time.perf_counter()
    bin_nside = CONFIG["BIN_NSIDE"] if bin_nside is None else int(bin_nside)
    theta_max_deg = theta_max_arcmin / 60.0
    ipix_all = hp.ang2pix(nside, ra_deg, dec_deg, nest=False, lonlat=True)
    ipix_ev = np.unique(ipix_all[from_target])
    if eval_mask is not None:
        ipix_ev = ipix_ev[eval_mask(ipix_ev)]
    n = int(ipix_ev.size)
    out = {k: np.full(n, np.nan) for k in ("ap_e", "sn_e", "ap_b", "sn_b")}
    out["counts"] = np.zeros(n, dtype=np.int32)
    out["sn_rand"] = np.full((n, n_random), np.nan)

    # Random rotations per galaxy (same draws as the per-galaxy notebook), then binned with the real shapes.
    g1r = g2r = None
    if n_random:
        rng = np.random.default_rng(seed)
        ang = rng.uniform(0.0, np.pi, size=(ra_deg.size, n_random))
        c2, s2 = np.cos(2 * ang), np.sin(2 * ang)
        g1r = g1[:, None] * c2 - g2[:, None] * s2
        g2r = g1[:, None] * s2 + g2[:, None] * c2
        del ang, c2, s2
    bins_ = bin_sources(bin_nside, ra_deg, dec_deg, g1, g2, w_gal, g1r, g2r, position=CONFIG["BIN_POSITION"])
    del g1r, g2r
    btree = bin_tree(bins_)

    it = tqdm(enumerate(ipix_ev), total=n, desc="HEALPix cells (binned)") if progress else enumerate(ipix_ev)
    for k, ip in it:
        # Cell center, then all bins within the search radius of it.
        ra_c, dec_c = hp.pix2ang(nside, int(ip), nest=False, lonlat=True)
        idx = btree.query_ball_point(radec_to_xyz(ra_c, dec_c)[0], r=chord_radius(theta_max_deg))
        if not idx:
            continue
        res = aperture_binned_at_center(
            float(ra_c), float(dec_c), bins_, idx, theta_max_arcmin=theta_max_arcmin, rs_deg=rs_deg, xc=xc,
            rand=bool(n_random),
        )
        out["ap_e"][k], out["sn_e"][k], out["ap_b"][k], out["sn_b"][k] = res[:4]
        if res[4] is not None:
            out["sn_rand"][k] = res[4]
        out["counts"][k] = res[5]
    out["ipix"] = ipix_ev
    out["completeness"] = out["counts"] / (N_DENSITY * np.pi * theta_max_arcmin**2)
    out["bins"] = bins_
    out["seconds"] = time.perf_counter() - t0
    return out


# Per-galaxy KD-tree on unit vectors, used by the per-galaxy comparisons in sections 7 and 8.
tree = cKDTree(radec_to_xyz(ra_deg, dec_deg))

theta_max_arcmin = CONFIG["THETA_MAX_ARCMIN"]
rs_deg = float(CONFIG["RS_DEG"])
nside = int(CONFIG["NSIDE"])

amap = run_aperture_map(nside, rs_deg, theta_max_arcmin, n_random=CONFIG["N_RANDOM"], seed=CONFIG["RANDOM_SEED"])
ipix_eval = amap["ipix"]
ap_mass_map, sn_ratio_map = amap["ap_e"], amap["sn_e"]
ap_bmode_map, sn_bmode_map = amap["ap_b"], amap["sn_b"]
counts_map, completeness_map, sn_rand_map = amap["counts"], amap["completeness"], amap["sn_rand"]
bins = amap["bins"]
n_cell = int(ipix_eval.size)

print(f"Source bins: {bins['ipix'].size:,} occupied NSIDE={bins['nside']} cells for {ra_deg.size:,} galaxies "
      f"(median {np.median(bins['n']):.0f}, mean {bins['n'].mean():.1f}, max {bins['n'].max()} per bin)")
print(f"Map time: {amap['seconds']:.1f} s with {CONFIG['N_RANDOM']} random realizations")
_good = completeness_map >= CONFIG["PEAK_COMPLETENESS_MIN"]
print("N HEALPix cells evaluated:", n_cell, f"({int(_good.sum())} with completeness >= {CONFIG['PEAK_COMPLETENESS_MIN']})")
print("Finite E-mode (aperture mass) cells:", int(np.isfinite(ap_mass_map).sum()))
print("Finite B-mode cells:", int(np.isfinite(ap_bmode_map).sum()))
print(f"std of S/N on complete cells:  E = {np.nanstd(sn_ratio_map[_good]):.2f}   B = {np.nanstd(sn_bmode_map[_good]):.2f}   "
      f"random rotations = {np.nanstd(sn_rand_map[_good]):.2f}")

### 4.4 Save the map

In [ ]:
# Sky coordinates of the evaluated cell centers.
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)

In [ ]:
aper = Table(
    data=[ra_pix, dec_pix, ap_mass_map, ap_bmode_map, sn_ratio_map, sn_bmode_map, completeness_map],
    names=["RA", "DEC", "E_ap", "B_ap", "E_SN", "B_SN", "COMPLETENESS"],
)
aper.meta.update({"RS_ARCMIN": CONFIG["RS_ARCMIN"], "FILTER_XC": CONFIG["FILTER_XC"], "NSIDE": nside, "SHEAR_FRAME": CONFIG["SHEAR_FRAME"], "GX_SIGN": "standard", "DEREDDEN": CONFIG["DEREDDEN"],
                  "BINNED": True, "BIN_NSIDE": CONFIG["BIN_NSIDE"], "BIN_POS": CONFIG["BIN_POSITION"], "BIN_WT": CONFIG["BIN_WEIGHT"]})
aper.write(OUTPUT_DIR / f"{CONFIG['TRACT']}_shapeHSM_massmap{RUN_TAG}.fits", format="fits", overwrite=True)

## 5.0 Peaks

A peak is a cell whose E-mode S/N is the highest of all evaluated cells within `PEAK_EXCL_RS_MULT · Rs`, whose aperture
is at least `PEAK_COMPLETENESS_MIN` complete, and whose S/N is at least `PEAK_SN_MIN`. The threshold is chosen to sit
above the highest noise peak found in the random-rotation maps. Those maps include shape noise only. Cosmic shear adds
about 20% to the map scatter at Rs = 15′ (section 7).

The completeness cut is 0.6. At 0.8 the cut removed Abell S 67 in tract 2877 (S/N ≈ 7, completeness 0.77 because its
aperture runs off the tract edge), while the B-mode map has no peak above S/N 4 down to completeness 0.5.

Each peak also carries B-mode information, which is never a cut: `B_FLAG` marks |S/N_B| ≥ `PEAK_B_SN_FLAG` at the peak,
and `B_PEAK_*` describe the nearest B-mode peak (either sign, |S/N_B| ≥ `PEAK_B_SN_FLAG`), with `B_PEAK_MATCH` set when it
lies within `PEAK_B_MATCH_RS_MULT · Rs`. The chance that a random position has a B-mode peak that close is printed and
stored in the table header (`B_CHANCE`), so a match is read as "look closer", not as a systematic. The peaks are written
to `output/<tract>_shapeHSM_peaks_binned.fits`. The full definition, with equations, is in `peak_definition.md`.

In [ ]:
sn_min = CONFIG.get("PEAK_SN_MIN")
excl_arcmin = CONFIG["PEAK_EXCL_RS_MULT"] * CONFIG["RS_ARCMIN"]
peak_kw = dict(nside=nside, excl_radius_arcmin=excl_arcmin, completeness_min=CONFIG["PEAK_COMPLETENESS_MIN"])

peak_ipix = find_peaks_shear_healpix(ipix_eval, sn_ratio_map, completeness_map, sn_min=sn_min, **peak_kw)
print("N peaks (with current cuts):", int(peak_ipix.size))

# Noise peaks in each random-rotation realization (no threshold), for the false-detection rate.
area_good = float(np.sum(completeness_map >= CONFIG["PEAK_COMPLETENESS_MIN"])) * hp.nside2pixarea(nside, degrees=True)
_idx_of = {int(p): k for k, p in enumerate(ipix_eval)}
rand_max, rand_n_above = [], []
for j in range(sn_rand_map.shape[1]):
    pk_r = find_peaks_shear_healpix(ipix_eval, sn_rand_map[:, j], completeness_map, sn_min=None, **peak_kw)
    vals = sn_rand_map[[_idx_of[int(p)] for p in pk_r], j]
    rand_max.append(np.nanmax(vals))
    rand_n_above.append(int(np.sum(vals >= (sn_min or 0))))
rand_max = np.asarray(rand_max)
print(f"Noise-only maps ({sn_rand_map.shape[1]} realizations, {area_good:.2f} deg^2 complete):")
print(f"  highest noise peak per tract: mean {rand_max.mean():.2f}, max {rand_max.max():.2f}")
print(f"  false peaks with S/N >= {sn_min}: {np.mean(rand_n_above):.2f} per tract = {np.mean(rand_n_above) / area_good:.2f} per deg^2")

# B-mode information for each peak (flags only; no peak is removed).
b_flag_sn = CONFIG["PEAK_B_SN_FLAG"]
b_match_arcmin = CONFIG["PEAK_B_MATCH_RS_MULT"] * CONFIG["RS_ARCMIN"]
bmatch = match_bmode_peaks(
    peak_ipix, ipix_eval, sn_bmode_map, completeness_map, nside=nside, excl_radius_arcmin=excl_arcmin,
    b_sn_min=b_flag_sn, match_radius_arcmin=b_match_arcmin, completeness_min=CONFIG["PEAK_COMPLETENESS_MIN"],
)
# Chance that a random position has a B-mode peak within the match radius (Poisson, from the B-mode peak density).
b_density = bmatch["b_ipix"].size / (area_good * 3600.0)  # per arcmin^2
b_chance = float(1.0 - np.exp(-b_density * np.pi * b_match_arcmin**2))
print(f"B-mode peaks with |S/N_B| >= {b_flag_sn}: {bmatch['b_ipix'].size} ({b_density * 3600.0:.1f} per deg^2); "
      f"chance of one within {b_match_arcmin:.1f}' of a random position = {b_chance:.2f}")

# Peak catalog, highest S/N first.
_k_pk = np.array([_idx_of[int(p)] for p in peak_ipix], dtype=np.int64)
_ra_pk, _dec_pk = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
peaks_tbl = Table(
    {
        "IPIX": peak_ipix,
        "RA": np.asarray(_ra_pk, dtype=np.float64),
        "DEC": np.asarray(_dec_pk, dtype=np.float64),
        "E_SN": sn_ratio_map[_k_pk],
        "B_SN": sn_bmode_map[_k_pk],
        "E_ap": ap_mass_map[_k_pk],
        "COMPLETENESS": completeness_map[_k_pk],
        "B_FLAG": np.abs(sn_bmode_map[_k_pk]) >= b_flag_sn,
        **{key: bmatch[key] for key in ("B_PEAK_SN", "B_PEAK_RA", "B_PEAK_DEC", "B_PEAK_SEP_ARCMIN", "B_PEAK_MATCH")},
    }
)
peaks_tbl.meta.update({
    "TRACT": CONFIG["TRACT"], "RS_ARCMIN": CONFIG["RS_ARCMIN"], "NSIDE": nside, "SN_MIN": sn_min,
    "COMP_MIN": CONFIG["PEAK_COMPLETENESS_MIN"], "EXCL_ARCMIN": excl_arcmin, "B_SN_FLAG": b_flag_sn,
    "B_MATCH_ARCMIN": b_match_arcmin, "B_DENSITY_DEG2": b_density * 3600.0, "B_CHANCE": b_chance,
    "DEREDDEN": CONFIG["DEREDDEN"], "BINNED": True, "BIN_NSIDE": CONFIG["BIN_NSIDE"], "BIN_POS": CONFIG["BIN_POSITION"], "BIN_WT": CONFIG["BIN_WEIGHT"],
})
peaks_tbl.write(OUTPUT_DIR / f"{CONFIG['TRACT']}_shapeHSM_peaks{RUN_TAG}.fits", format="fits", overwrite=True)

for row in peaks_tbl:
    _sep = ang_sep_deg(row["RA"], row["DEC"], (CLUSTER["ra"], CLUSTER["dec"])) * 60
    _bmatch = f"nearest B peak {row['B_PEAK_SN']:+.2f} at {row['B_PEAK_SEP_ARCMIN']:.1f}'" if np.isfinite(row["B_PEAK_SN"]) else "no B peak"
    print(f"  peak RA={row['RA']:.4f} Dec={row['DEC']:.4f}  S/N_E={row['E_SN']:.2f}  S/N_B={row['B_SN']:.2f}  "
          f"completeness={row['COMPLETENESS']:.2f}  {_sep:.1f}' from {CLUSTER['name']}  "
          f"B_FLAG={bool(row['B_FLAG'])}  {_bmatch}{' (match)' if row['B_PEAK_MATCH'] else ''}")
peaks_tbl

## 6.0 Figures

### 6.1 Plot setup

In [ ]:
# Recompute the cell centers so the figures can be run without the save step.
ra_pix, dec_pix = hp.pix2ang(nside, ipix_eval, nest=False, lonlat=True)
ra_pix = np.asarray(ra_pix, dtype=np.float64)
dec_pix = np.asarray(dec_pix, dtype=np.float64)

# Only plot cells with at least one galaxy in the aperture.
msk = counts_map > 0
if msk.any():
    m_lo, m_hi = np.nanpercentile(ap_mass_map[msk], [5, 95])
    sn_lo, sn_hi = np.nanpercentile(sn_ratio_map[msk], [5, 95])
    b_lo, b_hi = np.nanpercentile(ap_bmode_map[msk], [5, 95])
    snb_lo, snb_hi = np.nanpercentile(sn_bmode_map[msk], [5, 95])
else:
    m_lo = m_hi = sn_lo = sn_hi = b_lo = b_hi = snb_lo = snb_hi = None

# Marker size shrinks as the number of plotted cells grows.
pt_size = max(1.0, min(8.0, 500_000.0 / max(int(ipix_eval.size), 1)))

# gnomview needs a full-sky array, so skip it if that would be too large.
MAX_DENSE_NPIX = 4_000_000
_npix_hp = hp.nside2npix(nside)
do_gnomonic = _npix_hp <= MAX_DENSE_NPIX
if do_gnomonic:
    # Center the projection on the data.
    rot_ra = float(np.median(ra_pix))
    rot_dec = float(np.median(dec_pix))

    # Scatter the E-mode S/N onto the full sky, marking empty pixels as UNSEEN.
    sn_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    sn_dense[ipix_eval] = sn_ratio_map
    hp_sm = np.copy(sn_dense)
    hp_sm[~np.isfinite(hp_sm)] = hp.UNSEEN

    # Same for the B-mode S/N.
    snb_dense = np.full(_npix_hp, np.nan, dtype=np.float64)
    snb_dense[ipix_eval] = sn_bmode_map
    hp_smb = np.copy(snb_dense)
    hp_smb[~np.isfinite(hp_smb)] = hp.UNSEEN
else:
    hp_sm = hp_smb = None

### 6.2 E-mode maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)

# Left: aperture E-mode signal.
sc0 = axes[0].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=ap_mass_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=m_lo,
    vmax=m_hi,
    linewidths=0,
    alpha=0.85,
)
axes[0].set_xlabel("RA (deg)")
axes[0].set_ylabel("Dec (deg)")
axes[0].set_title(r"Aperture E-mode (gt×Q)")
axes[0].invert_xaxis()  # RA increases to the left
fig.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.02)

# Middle: the same quantity divided by its shape-noise error.
sc1 = axes[1].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-3.0,
    vmax=3.0,
    linewidths=0,
    alpha=0.85,
)
axes[1].set_xlabel("RA (deg)")
axes[1].set_ylabel("Dec (deg)")
axes[1].set_title(r"E-mode S/N")
axes[1].invert_xaxis()
fig.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.02)

# Right: source density per aperture, offset before the log to keep empty cells finite.
logc = np.log10(counts_map.astype(np.float64) + 0.1)
sc2 = axes[2].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=logc[msk],
    s=pt_size,
    cmap="magma",
    linewidths=0,
    alpha=0.85,
)
axes[2].set_xlabel("RA (deg)")
axes[2].set_ylabel("Dec (deg)")
axes[2].set_title(r"$\log_{10}$(counts in search disc) + 0.1")
axes[2].invert_xaxis()
fig.colorbar(sc2, ax=axes[2], fraction=0.046, pad=0.02)

for _ax in axes:
    mark_cluster(_ax, label=False)
mark_cluster(axes[1])

fig.suptitle(
    f"Tract {CONFIG['TRACT']}, NSIDE={nside}, Rs={CONFIG['RS_ARCMIN']:.2f}', search={theta_max_arcmin:.1f}', "
    f"binned at NSIDE={CONFIG['BIN_NSIDE']} ({CONFIG['BIN_WEIGHT']}, {CONFIG['BIN_POSITION']})",
    y=1.02,
)
save_plot(fig, f"{CONFIG['TRACT']}_E-mode_map")

### 6.3 B-mode maps

Structure here at the level of the E-mode maps would indicate systematics rather than lensing.

In [ ]:
fig_eb, axes_eb = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)

# Left: aperture B-mode signal.
scb0 = axes_eb[0].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=ap_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=b_lo,
    vmax=b_hi,
    linewidths=0,
    alpha=0.85,
)
axes_eb[0].set_xlabel("RA (deg)")
axes_eb[0].set_ylabel("Dec (deg)")
axes_eb[0].set_title(r"Aperture B-mode (g××Q)")
axes_eb[0].invert_xaxis()
fig_eb.colorbar(scb0, ax=axes_eb[0], fraction=0.046, pad=0.02)

# Right: B-mode S/N.
scb1 = axes_eb[1].scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_bmode_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-3.0,
    vmax=3.0,
    linewidths=0,
    alpha=0.85,
)
axes_eb[1].set_xlabel("RA (deg)")
axes_eb[1].set_ylabel("Dec (deg)")
axes_eb[1].set_title(r"B-mode S/N")
axes_eb[1].invert_xaxis()
fig_eb.colorbar(scb1, ax=axes_eb[1], fraction=0.046, pad=0.02)

for _ax in axes_eb:
    mark_cluster(_ax, label=False)

fig_eb.suptitle(
    f"Tract {CONFIG['TRACT']}, same binned filter/weights as the E-mode maps",
    y=1.02,
)
save_plot(fig_eb, f"{CONFIG['TRACT']}_B-mode_map")

In [ ]:
# E-mode against B-mode aperture values per occupied cell.
plt.hexbin(ap_mass_map[msk], ap_bmode_map[msk], mincnt=1)
plt.colorbar()
plt.gca().set_aspect(1)
save_plot(plt.gcf(), f"{CONFIG['TRACT']}_EB_hexbin")

In [ ]:
# Histogram of E-mode and B-mode aperture values per occupied cell.
# B should look like pure noise; an E-mode tail beyond it is the lensing signal.
e_vals = ap_mass_map[msk]
b_vals = ap_bmode_map[msk]
e_vals = e_vals[np.isfinite(e_vals)]
b_vals = b_vals[np.isfinite(b_vals)]
eb_bins = np.histogram_bin_edges(np.concatenate([e_vals, b_vals]), bins=60)

fig_hist, ax_hist = plt.subplots(figsize=(6, 4.2), constrained_layout=True)
ax_hist.hist(e_vals, bins=eb_bins, histtype="step", linewidth=1.5, label="E-mode")
ax_hist.hist(b_vals, bins=eb_bins, histtype="step", linewidth=1.5, label="B-mode")
ax_hist.set_xlabel("Aperture mass")
ax_hist.set_ylabel("Number of cells")
ax_hist.legend()
save_plot(fig_hist, f"{CONFIG['TRACT']}_EB_hist")

### 6.4 Peaks against PSZ2 G309.43-72.86

In [ ]:
fig2, axp = plt.subplots(figsize=(7, 5.5))

# E-mode S/N on a fixed -4 to 4 scale so peak significance is directly readable.
sc = axp.scatter(
    ra_pix[msk],
    dec_pix[msk],
    c=sn_ratio_map[msk],
    s=pt_size,
    cmap="RdBu_r",
    vmin=-4,
    vmax=4,
    linewidths=0,
    alpha=0.85,
)

# Circle the detected peaks.
if peak_ipix.size:
    ra_peak, dec_peak = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    axp.scatter(
        ra_peak,
        dec_peak,
        s=120,
        facecolors="none",
        edgecolors="k",
        linewidths=1.2,
        label="peaks",
    )
    print(ra_peak, dec_peak)

axp.set_xlabel("RA (deg)")
axp.set_ylabel("Dec (deg)")

# Planck SZ position of the target cluster.
mark_cluster(axp)
axp.grid()
axp.set_title(r"E-mode $\mathcal{S}/\mathcal{N}$ with peak positions")
axp.invert_xaxis()
fig2.colorbar(sc, ax=axp, fraction=0.046, pad=0.02)
axp.legend(loc="lower left", framealpha=1)
fig2.tight_layout()
save_plot(fig2, f"{CONFIG['TRACT']}_E-mode_sn_ratio")

### 6.5 Zoom on PSZ2 G309.43-72.86

In [ ]:
zoom_deg = 2.0 * CONFIG["RS_DEG"]  # Half-width of the zoom window
_cl_coord = SkyCoord(CLUSTER["ra"] * u.deg, CLUSTER["dec"] * u.deg)


def near_cluster(ra, dec):
    """Inside a box of half-width zoom_deg, in exact astropy offsets along RA and Dec from the cluster."""
    d_lon, d_lat = _cl_coord.spherical_offsets_to(SkyCoord(np.asarray(ra) * u.deg, np.asarray(dec) * u.deg))
    return (np.abs(d_lon.deg) < zoom_deg) & (np.abs(d_lat.deg) < zoom_deg)


in_zoom = msk & near_cluster(ra_pix, dec_pix)

# S/N of the cell holding the cluster position, if it was evaluated.
_ip_cl = hp.ang2pix(nside, CLUSTER["ra"], CLUSTER["dec"], nest=False, lonlat=True)
_k_cl = np.flatnonzero(ipix_eval == _ip_cl)
if _k_cl.size:
    print(f"E-mode S/N at cluster: {sn_ratio_map[_k_cl[0]]:.2f}  |  B-mode S/N: {sn_bmode_map[_k_cl[0]]:.2f}")
else:
    print("Cluster cell was not evaluated (no selected galaxy in it)")

fig_z, axz = plt.subplots(figsize=(6.5, 5.5))
scz = axz.scatter(
    ra_pix[in_zoom],
    dec_pix[in_zoom],
    c=sn_ratio_map[in_zoom],
    s=max(1.0, min(40.0, 200_000.0 / max(int(in_zoom.sum()), 1))),
    cmap="RdBu_r",
    vmin=-4,
    vmax=4,
    linewidths=0,
)

# Nearest detected peak to the cluster.
if peak_ipix.size:
    ra_peak, dec_peak = hp.pix2ang(nside, peak_ipix, nest=False, lonlat=True)
    _sep = ang_sep_deg(ra_peak, dec_peak, (CLUSTER["ra"], CLUSTER["dec"])) * 60.0
    _j = int(np.argmin(_sep))
    print(f"Nearest peak: RA={ra_peak[_j]:.4f}, Dec={dec_peak[_j]:.4f}, {_sep[_j]:.2f} arcmin from the cluster")
    _pz = near_cluster(ra_peak, dec_peak)
    axz.scatter(ra_peak[_pz], dec_peak[_pz], s=120, facecolors="none", edgecolors="k",
                linewidths=1.2, label="peaks")

mark_cluster(axz)
axz.set_xlabel("RA (deg)")
axz.set_ylabel("Dec (deg)")
axz.set_title(fr"E-mode $\mathcal{{S}}/\mathcal{{N}}$ around {CLUSTER['name']}")
axz.invert_xaxis()
axz.grid()
fig_z.colorbar(scz, ax=axz, fraction=0.046, pad=0.02)
axz.legend(loc="lower left", framealpha=1)
fig_z.tight_layout()
save_plot(fig_z, f"{CONFIG['TRACT']}_PSZ2_G309.43-72.86_zoom")

## 7.0 Filter and cell size

### 7.1 Analytic expectation

Consider a singular isothermal sphere (SIS) with Einstein radius θ_E, so that $\gamma_t(\theta) = \theta_E/2\theta$.
Averaging over all sources inside a circle of radius θ gives $\bar g_t(<\theta) = \theta_E/\theta$, with shape noise
$\sigma_g/\sqrt{n\pi\theta^2}$. The aperture S/N is then (Frieman 2026, eq. 5)

$$ \left(\frac{S}{N}\right)_{\rm top\text{-}hat} = \frac{\theta_E\sqrt{\pi n}}{\sigma_g}, $$

which does not depend on θ. A filter weighted as 1/θ, which the Schirmer filter is outside its core, reaches
$(S/N)^2 = (\pi n\theta_E^2/2\sigma_g^2)\ln(\theta_{\max}/\theta_{\min})$, so the S/N grows only logarithmically with aperture size.
For a real cluster this growth stops at large radius: the profile steepens beyond the NFW scale radius and r200, and
large-scale structure adds noise. At small radius, cluster members, blending, and strong lensing set the limit.

The S/N of a single map cell is, by comparison, $\gamma_t\sqrt{n A}/\sigma_g$. For a cell of area A = 1 arcmin² this is below 1
beyond a few arcmin of the cluster (computed below). The cell size therefore only sets where the statistic is sampled.
The detection S/N is set by the filter scale.

The next cell evaluates these expressions with the measured n_eff and σ_g, and fits θ_E (SIS) and the lensing efficiency
β (NFW, with the literature M200 and c = 4) to the profile from section 4.2.

In [ ]:
from astropy.cosmology import FlatLambdaCDM
import astropy.constants as const
from scipy.optimize import curve_fit

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
D_L = cosmo.angular_diameter_distance(CLUSTER["z"])
ARCMIN_MPC = (D_L * (1 * u.arcmin).to(u.rad).value).to(u.Mpc).value
rho_c = cosmo.critical_density(CLUSTER["z"]).to(u.Msun / u.Mpc**3).value
R200_MPC = (3 * CLUSTER["M200"] / (4 * np.pi * 200 * rho_c)) ** (1 / 3)
print(f"z = {CLUSTER['z']}: 1' = {ARCMIN_MPC:.3f} Mpc;  r200 = {R200_MPC:.2f} Mpc = {R200_MPC / ARCMIN_MPC:.1f}'")


def nfw_gt(theta_arcmin, beta, M200=CLUSTER["M200"], c200=CLUSTER["c200"]):
    """NFW tangential shear (Wright & Brainerd 2000) for sources with lensing efficiency beta = <D_LS/D_S>."""
    r200 = (3 * M200 / (4 * np.pi * 200 * rho_c)) ** (1 / 3)
    rs = r200 / c200
    dc = 200 / 3 * c200**3 / (np.log(1 + c200) - c200 / (1 + c200))
    x = np.atleast_1d(np.asarray(theta_arcmin, float) * ARCMIN_MPC / rs)
    gfn = np.empty_like(x)
    lo, hi = x < 1, x > 1
    a = np.arctanh(np.sqrt((1 - x[lo]) / (1 + x[lo])))
    gfn[lo] = (8 * a / (x[lo] ** 2 * np.sqrt(1 - x[lo] ** 2)) + 4 / x[lo] ** 2 * np.log(x[lo] / 2)
               - 2 / (x[lo] ** 2 - 1) + 4 * a / ((x[lo] ** 2 - 1) * np.sqrt(1 - x[lo] ** 2)))
    b = np.arctan(np.sqrt((x[hi] - 1) / (1 + x[hi])))
    gfn[hi] = (8 * b / (x[hi] ** 2 * np.sqrt(x[hi] ** 2 - 1)) + 4 / x[hi] ** 2 * np.log(x[hi] / 2)
               - 2 / (x[hi] ** 2 - 1) + 4 * b / (x[hi] ** 2 - 1) ** 1.5)
    gfn[~(lo | hi)] = 10 / 3 + 4 * np.log(0.5)
    sig_crit = (const.c**2 / (4 * np.pi * const.G) / (D_L * beta)).to(u.Msun / u.Mpc**2).value
    return rs * dc * rho_c * gfn / sig_crit


# Fit the profile from 4.2 at theta > 1.5', outside the member-contaminated core.
rc, mt, mx, err = prof
fit = rc > 1.5
(thE,), cov = curve_fit(lambda t, a: a / (2 * t), rc[fit], mt[fit], sigma=err[fit], p0=[0.3], absolute_sigma=True)
(beta_fit,), covb = curve_fit(nfw_gt, rc[fit], mt[fit], sigma=err[fit], p0=[0.5], absolute_sigma=True)
print(f"SIS fit: theta_E = {thE:.3f} ± {np.sqrt(cov[0, 0]):.3f} arcmin")
for zs in (0.8, 1.0, 1.2):
    beta = (cosmo.angular_diameter_distance_z1z2(CLUSTER["z"], zs) / cosmo.angular_diameter_distance(zs)).value
    print(f"   -> sigma_v = {np.sqrt(np.deg2rad(thE / 60) / (4 * np.pi * beta)) * 3e5:.0f} km/s if all sources were at z_s = {zs} (beta = {beta:.2f})")
print(f"NFW fit with M200 = {CLUSTER['M200']:.1e}, c = {CLUSTER['c200']}: effective beta = {beta_fit:.2f} ± {np.sqrt(covb[0, 0]):.2f}")
print("   (beta below ~0.55 for z_s ~ 1 indicates dilution by foreground and cluster-member galaxies: there is no photo-z cut)")

print("\nAnalytic S/N with the measured n_eff and sigma_g:")
sn_tophat = thE * np.sqrt(np.pi * N_EFF) / SIGMA_G
print(f"   theta_E sqrt(pi n_eff)/sigma_g = {thE:.3f} x sqrt(pi x {N_EFF:.1f}) / {SIGMA_G:.3f} = {sn_tophat:.1f}  (top-hat, any radius)")
print(f"   per 1 arcmin^2 cell at theta = 2': S/N = {thE / 4 * np.sqrt(N_EFF) / SIGMA_G:.2f}")

### 7.2 S/N against filter scale

For each aperture radius Rs, and three values of the core radius x_c, we compute:

* **measured**: the binned E-mode S/N at the SZ position, using the binned aperture function from the map, with the
  per-galaxy value (section 1.3) alongside for x_c = 0.15;
* **predicted**: the same statistic with $g_t$ replaced by the fitted NFW (or SIS) shear, with the noise term computed from
  the real source positions, weights, and ellipticities, so masks and depth variations are included;
* **noise peaks**: the highest peak in the tract in random-rotation noise maps, using NSIDE = 1024 for speed. Coarse cells
  undersample the peaks, so these values run about 0.5 below the NSIDE = 4096 values in section 5. They are for comparing
  one Rs with another.

S/N values at neighboring Rs are strongly correlated, and the cluster S/N has a statistical scatter of about ±1.

The bins stay at `BIN_NSIDE` for every Rs. At small Rs the bins are no longer small compared with the filter
(0.86′ is 0.43 Rs at Rs = 2′), so the gap between binned and per-galaxy values there measures the binning error.

In [ ]:
RS_GRID = np.array([2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25])
XC_GRID = [0.05, 0.15, 0.3]
_cdeg = (CLUSTER["ra"], CLUSTER["dec"])
_idx_cl = np.asarray(tree.query_ball_point(radec_to_xyz(*_cdeg)[0], r=chord_radius(1.25 * RS_GRID.max() / 60)))


# Bins of the map (4.3) near the cluster.
_btree = bin_tree(bins)
_bidx_cl = np.asarray(_btree.query_ball_point(radec_to_xyz(*_cdeg)[0], r=chord_radius(1.25 * RS_GRID.max() / 60)))


def sn_at_cluster(rs_arcmin, xc):
    """Binned (ap_e, sn_e, ap_b, sn_b) at the SZ position, with the kernel of the map."""
    return aperture_binned_at_center(*_cdeg, bins, _bidx_cl, theta_max_arcmin=1.2 * rs_arcmin,
                                     rs_deg=rs_arcmin / 60, xc=xc)[:4]


def sn_at_cluster_gal(rs_arcmin, xc, g1_=None, g2_=None):
    """Per-galaxy (ap_e, sn_e, ap_b, sn_b, None) at the SZ position (section 1.3)."""
    g1_ = g1 if g1_ is None else g1_
    g2_ = g2 if g2_ is None else g2_
    return aperture_mass_and_sn_at_center(*_cdeg, ra_deg[_idx_cl], dec_deg[_idx_cl], g1_[_idx_cl], g2_[_idx_cl],
                                          theta_max_arcmin=1.2 * rs_arcmin, rs_deg=rs_arcmin / 60, w_all=w_gal[_idx_cl], xc=xc)


def predicted_sn(rs_arcmin, xc, model):
    """S/N of a noiseless model shear field sampled at the real source positions with the real noise denominator."""
    r = ang_sep_deg(ra_deg[_idx_cl], dec_deg[_idx_cl], _cdeg) * 60
    s = r <= 1.2 * rs_arcmin
    q = schirmer_weight(r[s], rs_arcmin, xc)
    ws = w_gal[_idx_cl][s]
    den = np.sqrt(np.sum(q**2 * ws**2 * (g1[_idx_cl][s] ** 2 + g2[_idx_cl][s] ** 2)) / 2)
    return np.sum(q * ws * model(np.maximum(r[s], 0.05))) / den


sweep = {xc: np.array([sn_at_cluster(rs, xc)[1] for rs in RS_GRID]) for xc in XC_GRID}
sweep_B = np.array([sn_at_cluster(rs, CONFIG["FILTER_XC"])[3] for rs in RS_GRID])
sweep_gal = np.array([sn_at_cluster_gal(rs, CONFIG["FILTER_XC"])[1] for rs in RS_GRID])
pred_nfw = np.array([predicted_sn(rs, CONFIG["FILTER_XC"], lambda t: nfw_gt(t, beta_fit)) for rs in RS_GRID])
pred_sis = np.array([predicted_sn(rs, CONFIG["FILTER_XC"], lambda t: thE / (2 * t)) for rs in RS_GRID])

# Coarse noise-only maps per Rs: highest random-rotation peak in the tract (shape noise only).
NSIDE_SWEEP, N_RAND_SWEEP = 1024, 10
noise_max = []
for rs in tqdm(RS_GRID, desc="noise maps per Rs"):
    nm = run_aperture_map(NSIDE_SWEEP, rs / 60, 1.2 * rs, n_random=N_RAND_SWEEP, seed=1, progress=False)
    _kw = dict(nside=NSIDE_SWEEP, excl_radius_arcmin=CONFIG["PEAK_EXCL_RS_MULT"] * rs, completeness_min=CONFIG["PEAK_COMPLETENESS_MIN"])
    _io = {int(p): k for k, p in enumerate(nm["ipix"])}
    mx_ = []
    for j in range(N_RAND_SWEEP):
        pk_ = find_peaks_shear_healpix(nm["ipix"], nm["sn_rand"][:, j], nm["completeness"], **_kw)
        mx_.append(np.nanmax(nm["sn_rand"][[_io[int(p)] for p in pk_], j]) if pk_.size else np.nan)
    good_ = nm["completeness"] >= CONFIG["PEAK_COMPLETENESS_MIN"]
    _ra_n, _dec_n = hp.pix2ang(NSIDE_SWEEP, nm["ipix"], lonlat=True)
    _sep_n = ang_sep_deg(_ra_n, _dec_n, _cdeg) * 60
    far_ = good_ & (_sep_n > max(2 * rs, 10))
    noise_max.append((np.nanmean(mx_), np.nanpercentile(mx_, 90), np.nanstd(nm["sn_e"][far_]) if far_.sum() > 20 else np.nan))
noise_max = np.array(noise_max)

print("Binned S/N, with the per-galaxy S/N at x_c = 0.15 for comparison:")
print(" Rs[']  S/N(xc=.05)  S/N(xc=.15)  S/N(xc=.3)  per-gal(xc=.15)  B(xc=.15)  pred_NFW  pred_SIS  noise-max(mean/90%)  std(E far)")
for i, rs in enumerate(RS_GRID):
    print(f"{rs:6.0f}  {sweep[0.05][i]:10.2f}  {sweep[0.15][i]:11.2f}  {sweep[0.3][i]:10.2f}  {sweep_gal[i]:15.2f}  {sweep_B[i]:9.2f}  "
          f"{pred_nfw[i]:8.2f}  {pred_sis[i]:8.2f}  {noise_max[i, 0]:9.2f} / {noise_max[i, 1]:.2f}  {noise_max[i, 2]:9.2f}")

In [ ]:
fig_s, ax_s = plt.subplots(figsize=(8, 5))
for xc, ls in zip(XC_GRID, [":", "-", "--"]):
    ax_s.plot(RS_GRID, sweep[xc], "o" + ls, color="C3", ms=4 if xc != CONFIG["FILTER_XC"] else 6,
              alpha=1 if xc == CONFIG["FILTER_XC"] else 0.5, label=f"binned, measured at SZ position, $x_c$={xc}")
ax_s.plot(RS_GRID, sweep_gal, "x-", color="C4", lw=1, label=f"per-galaxy, measured at SZ position, $x_c$={CONFIG['FILTER_XC']}")
ax_s.plot(RS_GRID, pred_nfw, "-", color="k", lw=2, label=fr"NFW prediction (M200=1.4e15, fitted $\beta$={beta_fit:.2f})")
ax_s.plot(RS_GRID, pred_sis, "--", color="0.5", label=fr"SIS prediction ($\theta_E$={thE:.2f}')")
ax_s.axhline(sn_tophat, color="C0", ls="-.", lw=1, label="Frieman eq. (5), measured n_eff, σ_g")
ax_s.fill_between(RS_GRID, 0, noise_max[:, 1], color="0.85", label="highest noise peak per tract (90th pct)")
ax_s.plot(RS_GRID, noise_max[:, 0], color="0.6", lw=1)
ax_s.plot(RS_GRID, sweep_B, "s", mfc="none", color="0.4", label="B-mode at SZ position")
ax_s.axvline(CONFIG["RS_ARCMIN"], color="C2", lw=1.5, label=f"chosen Rs = {CONFIG['RS_ARCMIN']:.0f}'")
ax_s.axvline(R200_MPC / ARCMIN_MPC, color="C1", lw=1, ls=":", label="r200")
ax_s.set_xscale("log")
ax_s.set_xticks(RS_GRID)
ax_s.set_xticklabels([str(r) for r in RS_GRID])
ax_s.set_xlabel("Schirmer aperture radius Rs (arcmin)")
ax_s.set_ylabel("S/N")
ax_s.set_ylim(-1, 10)
ax_s.legend(fontsize=8, loc="lower right", framealpha=1)
ax_s.set_title(f"{CLUSTER['name']}: binned S/N against filter scale")
fig_s.tight_layout()
save_plot(fig_s, f"{CONFIG['TRACT']}_PSZ2_G309_sn_vs_rs")

### 7.3 Choice of filter and cell size

*The numbers quoted in this section come from the per-galaxy notebook (`dp2_shearmap_PSZ2_G309.43-72.86`). Section 8 compares the binned estimator with them.*

We measure aperture-mass statistics with the Schirmer et al. (2007) filter. It is flat inside a core of radius x_c Rs, falls
as 1/θ outside it (the matched filter for an isothermal shear profile), and is truncated at θ ≈ Rs. Two scales are involved.
The filter radius Rs sets the signal-to-noise. The HEALPix cell size sets only how finely the S/N field is sampled. We choose
them separately.

**Filter scale.** For an isothermal lens, the S/N of a matched aperture grows only as $\sqrt{\ln(R_s/\theta_{\min})}$, and for
an NFW lens it saturates once the aperture extends past r200. With the measured source density
(n_eff ≈ 18 arcmin⁻²) and per-component shape noise (σ_g ≈ 0.34), the NFW profile fitted to the tangential shear of
PSZ2 G309.43-72.86 predicts an S/N that rises steeply up to Rs ≈ r200 ≈ 7′ and stays within 5% of its maximum for
Rs ≈ 11–25′. The measured S/N at the cluster follows the same curve and is highest at Rs = 15–20′ (S/N ≈ 7.5–7.8).
Larger apertures add noise from uncorrelated large-scale structure. The scatter of the E-mode S/N map away from the
cluster grows from 1.2 at Rs = 15′ to 1.45 at 20′, compared with 1.0 for shape noise alone. Larger apertures also
lose more area at survey edges and blend neighboring halos. We therefore adopt **Rs = 15′** (≈ 2.2 r200). The core radius
has little effect: x_c from 0.05 to 0.3 changes the S/N by less than 0.5. We keep the NFW-optimized value
**x_c = 0.15**. Because the optimal aperture scales with r200 ∝ M^{1/3}, a blind search for lower-mass clusters
(M200 ~ 3×10¹⁴ M⊙, r200 ~ 4′ at z ≈ 0.3) would use Rs ≈ 6–10′.

**Cell size.** The S/N peak at the cluster is sharp, with a half-width of about 1.5′ set by the filter core. It must be
sampled on a grid fine enough that some cell center falls near the maximum. Cell spacing s gives a worst-case
offset of about s/√2. On cells of NSIDE = 4096 (0.86′) this costs at most ≈ 0.7σ and typically ≈ 0.2σ. NSIDE = 2048 and
1024 cost up to ≈ 1.8σ and ≈ 3.7σ, and NSIDE = 8192 gains only ≈ 0.2σ for four times the cost. We adopt **NSIDE = 4096**.

**Peak definition.** Peaks are maxima of the S/N map, not of M_ap, so that regions with fewer sources do not produce
spuriously large M_ap values. The exclusion radius is 0.4 Rs, where the autocorrelation of the noise maps drops to about
0.1 (half-width ≈ 0.2 Rs). Apertures must be at least 60% complete (80% removed Abell S 67, whose aperture runs off the tract edge; section 5.0). The threshold is S/N ≥ 4, above the highest peak in every
random-rotation realization of the tract (mean highest peak ≈ 3.4).

With these choices, PSZ2 G309.43-72.86 is the highest peak in the tract (S/N ≈ 7.7, 0.2′ from the SZ position). This agrees
with the analytic expectation once the measured n_eff and σ_g are used.

## 8.0 Binned against per-galaxy

Three checks of the binned estimator against the per-galaxy one of section 1.3, with the same sources, weights,
filter, and evaluation cells:

1. **Cell by cell** on a random subset of complete cells plus the peak and cluster cells (8.1). The per-galaxy kernel is
   run directly here, so the comparison is always like for like. Moving each source to its bin changes its polar
   angle, and so its g_t and its share of the shape noise, so the differences scatter from cell to cell even without
   a bias. Compare that scatter with the map noise (std of the random-rotation S/N ≈ 1), and look for a systematic
   offset at the peaks, where the signal is.
2. **Full map and peaks** against the per-galaxy notebook's saved `output/<tract>_shapeHSM_massmap.fits` and
   `_peaks.fits`, if they exist and were made with the same Rs, x_c, NSIDE, and dust setting (8.2).
3. **Bin size** at the cluster and the peaks, from coarse bins to bins smaller than the mean source spacing (8.3).
   As the bins shrink the binned S/N must converge to the per-galaxy value.

### 8.1 Cell by cell

In [ ]:
N_COMPARE = 400  # Random complete cells compared, on top of the peak and cluster cells
_rng_cmp = np.random.default_rng(0)
_good_k = np.flatnonzero(completeness_map >= CONFIG["PEAK_COMPLETENESS_MIN"])
_k_special = np.unique(np.concatenate([np.asarray(_k_pk, dtype=np.int64), np.asarray(_k_cl, dtype=np.int64)]))
k_cmp = np.unique(np.concatenate([
    _rng_cmp.choice(_good_k, size=min(N_COMPARE, _good_k.size), replace=False), _k_special,
]))
_btree_map = bin_tree(bins)

cmp = {key: np.full(k_cmp.size, np.nan) for key in ("sn_e_gal", "sn_b_gal", "ap_e_gal", "sn_e_bin", "sn_b_bin", "ap_e_bin")}
t_gal = t_bin = 0.0
for n_, k in enumerate(tqdm(k_cmp, desc="per-galaxy vs binned")):
    ra_c, dec_c = float(ra_pix[k]), float(dec_pix[k])
    q = radec_to_xyz(ra_c, dec_c)[0]
    t0 = time.perf_counter()
    idx = np.asarray(tree.query_ball_point(q, r=chord_radius(theta_max_deg)), dtype=np.int64)
    rg = aperture_mass_and_sn_at_center(ra_c, dec_c, ra_deg[idx], dec_deg[idx], g1[idx], g2[idx],
                                        theta_max_arcmin=theta_max_arcmin, rs_deg=rs_deg, w_all=w_gal[idx])
    t1 = time.perf_counter()
    rb = aperture_binned_at_center(ra_c, dec_c, bins, _btree_map.query_ball_point(q, r=chord_radius(theta_max_deg)),
                                   theta_max_arcmin=theta_max_arcmin, rs_deg=rs_deg)
    t2 = time.perf_counter()
    t_gal += t1 - t0
    t_bin += t2 - t1
    cmp["ap_e_gal"][n_], cmp["sn_e_gal"][n_], cmp["sn_b_gal"][n_] = rg[0], rg[1], rg[3]
    cmp["ap_e_bin"][n_], cmp["sn_e_bin"][n_], cmp["sn_b_bin"][n_] = rb[0], rb[1], rb[3]

# The binned values here must equal the map values (same kernel, same bins).
assert np.allclose(cmp["sn_e_bin"], sn_ratio_map[k_cmp], equal_nan=True)

d_e = cmp["sn_e_bin"] - cmp["sn_e_gal"]
d_b = cmp["sn_b_bin"] - cmp["sn_b_gal"]
print(f"{k_cmp.size} cells, binned at NSIDE={CONFIG['BIN_NSIDE']} ({CONFIG['BIN_WEIGHT']}, {CONFIG['BIN_POSITION']})")
for name, d in (("S/N_E", d_e), ("S/N_B", d_b)):
    print(f"  Δ{name} = binned - per-galaxy: mean {np.nanmean(d):+.4f}, std {np.nanstd(d):.4f}, "
          f"max |Δ| {np.nanmax(np.abs(d)):.4f}")
print(f"  correlation of S/N_E: {np.corrcoef(cmp['sn_e_bin'], cmp['sn_e_gal'])[0, 1]:.5f}")
print(f"  time per cell (no randoms): per-galaxy {1e3 * t_gal / k_cmp.size:.1f} ms, binned {1e3 * t_bin / k_cmp.size:.1f} ms "
      f"(x{t_gal / max(t_bin, 1e-12):.1f})")

print("\nPeaks and the cluster cell:")
for k in _k_special:
    j = int(np.flatnonzero(k_cmp == k)[0])
    print(f"  RA={ra_pix[k]:.4f} Dec={dec_pix[k]:.4f}: S/N_E binned {cmp['sn_e_bin'][j]:.3f}, per-galaxy {cmp['sn_e_gal'][j]:.3f}  |  "
          f"S/N_B binned {cmp['sn_b_bin'][j]:+.3f}, per-galaxy {cmp['sn_b_gal'][j]:+.3f}")

In [ ]:
fig_c, axes_c = plt.subplots(1, 3, figsize=(15, 4.3), constrained_layout=True)
for ax, key, lab in ((axes_c[0], "sn_e", r"E-mode $\mathcal{S}/\mathcal{N}$"), (axes_c[1], "sn_b", r"B-mode $\mathcal{S}/\mathcal{N}$")):
    x_, y_ = cmp[f"{key}_gal"], cmp[f"{key}_bin"]
    lim = np.nanmax(np.abs(np.concatenate([x_, y_]))) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "k-", lw=0.8)
    ax.scatter(x_, y_, s=6, color="C0", label="random complete cells")
    _sp = np.isin(k_cmp, _k_special)
    ax.scatter(x_[_sp], y_[_sp], s=60, facecolors="none", edgecolors="C3", label="peaks / cluster")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect(1)
    ax.set_xlabel(f"per-galaxy {lab}")
    ax.set_ylabel(f"binned {lab}")
    ax.legend(loc="upper left", fontsize=9)
axes_c[2].hist(d_e, bins=40, histtype="step", lw=1.5, label=r"$\Delta\mathcal{S}_E$")
axes_c[2].hist(d_b, bins=40, histtype="step", lw=1.5, label=r"$\Delta\mathcal{S}_B$")
axes_c[2].set_xlabel("binned - per-galaxy S/N")
axes_c[2].set_ylabel("cells")
axes_c[2].legend()
fig_c.suptitle(f"Tract {CONFIG['TRACT']}: binned (NSIDE={CONFIG['BIN_NSIDE']}) vs per-galaxy aperture S/N, Rs={CONFIG['RS_ARCMIN']:.0f}'")
save_plot(fig_c, f"{CONFIG['TRACT']}_vs_galaxy_cells")

### 8.2 Full map and peaks against the per-galaxy notebook

Reads the map and peak catalog that `dp2_shearmap_PSZ2_G309.43-72.86.ipynb` writes to `output/`. The comparison is
skipped when the files are missing or their headers show different Rs, x_c, NSIDE, or dust settings: rerun the
per-galaxy notebook first in that case. Source selection changes do not show up in the headers, so make sure both
notebooks were run with the same CONFIG.

In [ ]:
_gal_map_path = OUTPUT_DIR / f"{CONFIG['TRACT']}_shapeHSM_massmap.fits"
_gal_pk_path = OUTPUT_DIR / f"{CONFIG['TRACT']}_shapeHSM_peaks.fits"


def _settings_match(meta, keys=("RS_ARCMIN", "FILTER_XC", "NSIDE", "DEREDDEN")):
    """Header keys that differ from this run, as {key: (file, this run)}."""
    this_run = {"RS_ARCMIN": CONFIG["RS_ARCMIN"], "FILTER_XC": CONFIG["FILTER_XC"], "NSIDE": nside, "DEREDDEN": CONFIG["DEREDDEN"]}
    bad = {}
    for key in keys:
        val = this_run[key]
        got = meta.get(key)
        if got is None or (np.isscalar(val) and not isinstance(val, bool) and not np.isclose(float(got), float(val))) \
                or (isinstance(val, bool) and bool(got) != val):
            bad[key] = (got, val)
    return bad


if not _gal_map_path.exists():
    print(f"{_gal_map_path.name} not found: run dp2_shearmap_PSZ2_G309.43-72.86.ipynb first")
else:
    gal_map = Table.read(_gal_map_path)
    _bad = _settings_match(gal_map.meta)
    if _bad:
        print(f"{_gal_map_path.name} was made with different settings, skipping (file, this run): {_bad}")
    else:
        _ip_gal = hp.ang2pix(nside, np.asarray(gal_map["RA"]), np.asarray(gal_map["DEC"]), nest=False, lonlat=True)
        common, i_bin, i_gal = np.intersect1d(ipix_eval, _ip_gal, return_indices=True)
        print(f"{common.size:,} common cells ({ipix_eval.size:,} binned, {_ip_gal.size:,} per-galaxy)")
        _cgood = completeness_map[i_bin] >= CONFIG["PEAK_COMPLETENESS_MIN"]
        for name, b_, g_ in (("S/N_E", sn_ratio_map, "E_SN"), ("S/N_B", sn_bmode_map, "B_SN"), ("E_ap", ap_mass_map, "E_ap")):
            d = b_[i_bin][_cgood] - np.asarray(gal_map[g_])[i_gal][_cgood]
            scale = np.nanstd(np.asarray(gal_map[g_])[i_gal][_cgood])
            print(f"  Δ{name} on complete cells: mean {np.nanmean(d):+.3e}, std {np.nanstd(d):.3e} "
                  f"({np.nanstd(d) / scale:.3f} of the map std), max |Δ| {np.nanmax(np.abs(d)):.3e}")

        fig_m, axes_m = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
        _dsn = sn_ratio_map[i_bin] - np.asarray(gal_map["E_SN"])[i_gal]
        _lim = np.nanpercentile(np.abs(_dsn), 99)
        scm = axes_m[0].scatter(ra_pix[i_bin], dec_pix[i_bin], c=_dsn, s=pt_size, cmap="RdBu_r", vmin=-_lim, vmax=_lim, linewidths=0)
        axes_m[0].invert_xaxis()
        axes_m[0].set_xlabel("RA (deg)")
        axes_m[0].set_ylabel("Dec (deg)")
        axes_m[0].set_title(r"binned - per-galaxy E-mode $\mathcal{S}/\mathcal{N}$")
        mark_cluster(axes_m[0], label=False)
        fig_m.colorbar(scm, ax=axes_m[0], fraction=0.046, pad=0.02)
        axes_m[1].scatter(completeness_map[i_bin], _dsn, s=2, color="0.3")
        axes_m[1].axvline(CONFIG["PEAK_COMPLETENESS_MIN"], color="C3", ls="--", lw=1)
        axes_m[1].set_xlabel("aperture completeness")
        axes_m[1].set_ylabel(r"$\Delta\mathcal{S}_E$")
        save_plot(fig_m, f"{CONFIG['TRACT']}_vs_galaxy_map")

if _gal_pk_path.exists():
    gal_pk = Table.read(_gal_pk_path)
    _bad = _settings_match(gal_pk.meta, keys=("RS_ARCMIN", "NSIDE", "DEREDDEN"))  # the peak header has no FILTER_XC
    if _bad:
        print(f"{_gal_pk_path.name} was made with different settings, skipping (file, this run): {_bad}")
    else:
        print(f"\nPeaks: {len(peaks_tbl)} binned, {len(gal_pk)} per-galaxy (match radius {excl_arcmin:.1f}')")
        for row in gal_pk:
            sep = ang_sep_deg(peaks_tbl["RA"], peaks_tbl["DEC"], (row["RA"], row["DEC"])) * 60 if len(peaks_tbl) else np.array([])
            if sep.size and sep.min() <= excl_arcmin:
                j = int(np.argmin(sep))
                print(f"  per-galaxy RA={row['RA']:.4f} Dec={row['DEC']:.4f} S/N={row['E_SN']:.2f}  ->  binned S/N={peaks_tbl['E_SN'][j]:.2f} "
                      f"at {sep[j]:.2f}'")
            else:
                print(f"  per-galaxy RA={row['RA']:.4f} Dec={row['DEC']:.4f} S/N={row['E_SN']:.2f}  ->  no binned peak within {excl_arcmin:.1f}'")
        for j, row in enumerate(peaks_tbl):
            sep = ang_sep_deg(gal_pk["RA"], gal_pk["DEC"], (row["RA"], row["DEC"])) * 60 if len(gal_pk) else np.array([])
            if not sep.size or sep.min() > excl_arcmin:
                print(f"  binned only: RA={row['RA']:.4f} Dec={row['DEC']:.4f} S/N={row['E_SN']:.2f}")
else:
    print(f"{_gal_pk_path.name} not found: run dp2_shearmap_PSZ2_G309.43-72.86.ipynb first")

### 8.3 Bin size

The binned S/N at the cluster and at each peak for a range of `BIN_NSIDE`, with bins at the cell center and at the
weighted centroid, against the per-galaxy value. The per-galaxy kernel is the limit of very small bins. The map
itself is not recomputed here; only the apertures at these positions are.

In [ ]:
BIN_NSIDE_GRID = [512, 1024, 2048, 4096, 8192, 16384]
_pos_names = [CLUSTER["name"]] + [f"peak {j + 1}" for j in range(len(peaks_tbl))]
_pos = [(CLUSTER["ra"], CLUSTER["dec"])] + [(float(r["RA"]), float(r["DEC"])) for r in peaks_tbl]
_xyz_pos = radec_to_xyz([p[0] for p in _pos], [p[1] for p in _pos])

# Per-galaxy reference at each position.
sn_ref = []
for (ra_c, dec_c), q in zip(_pos, _xyz_pos):
    idx = np.asarray(tree.query_ball_point(q, r=chord_radius(theta_max_deg)), dtype=np.int64)
    sn_ref.append(aperture_mass_and_sn_at_center(ra_c, dec_c, ra_deg[idx], dec_deg[idx], g1[idx], g2[idx],
                                                 theta_max_arcmin=theta_max_arcmin, rs_deg=rs_deg, w_all=w_gal[idx])[1])
sn_ref = np.array(sn_ref)

sn_grid = {p: np.full((len(BIN_NSIDE_GRID), len(_pos)), np.nan) for p in ("center", "centroid")}
bin_info = []
for a, nsb in enumerate(BIN_NSIDE_GRID):
    for p in ("center", "centroid"):
        b_ = bin_sources(nsb, ra_deg, dec_deg, g1, g2, w_gal, position=p)
        t_ = bin_tree(b_)
        for c, ((ra_c, dec_c), q) in enumerate(zip(_pos, _xyz_pos)):
            sn_grid[p][a, c] = aperture_binned_at_center(ra_c, dec_c, b_, t_.query_ball_point(q, r=chord_radius(theta_max_deg)),
                                                         theta_max_arcmin=theta_max_arcmin, rs_deg=rs_deg)[1]
    bin_info.append((np.rad2deg(hp.nside2resol(nsb)) * 60, b_["ipix"].size, ra_deg.size / b_["ipix"].size))

print(f"Binned E-mode S/N ({CONFIG['BIN_WEIGHT']} weights); per-galaxy: " +
      ", ".join(f"{n} {s:.3f}" for n, s in zip(_pos_names, sn_ref)))
print(f"{'NSIDE':>6} {'bin [arcmin]':>12} {'bin/Rs':>7} {'gal/bin':>8}  " + "  ".join(f"{n[:23]:>23}" for n in _pos_names))
print(f"{'':>36}  " + "  ".join(f"{'center':>11} {'centroid':>11}" for _ in _pos_names))
for a, nsb in enumerate(BIN_NSIDE_GRID):
    size, nb_, gpb = bin_info[a]
    print(f"{nsb:>6} {size:>12.2f} {size / CONFIG['RS_ARCMIN']:>7.3f} {gpb:>8.1f}  " +
          "  ".join(f"{sn_grid['center'][a, c]:>11.3f} {sn_grid['centroid'][a, c]:>11.3f}" for c in range(len(_pos))))

In [ ]:
fig_b, ax_b = plt.subplots(figsize=(7.5, 4.5))
_sizes = np.array([b[0] for b in bin_info])
for c, name in enumerate(_pos_names):
    col = f"C{c}"
    ax_b.plot(_sizes, sn_grid["center"][:, c] - sn_ref[c], "o-", color=col, label=f"{name}, center")
    ax_b.plot(_sizes, sn_grid["centroid"][:, c] - sn_ref[c], "s--", mfc="none", color=col, label=f"{name}, centroid")
ax_b.axhline(0, color="k", lw=0.8)
ax_b.axvline(np.rad2deg(hp.nside2resol(CONFIG["BIN_NSIDE"])) * 60, color="0.5", ls=":", label=f"BIN_NSIDE = {CONFIG['BIN_NSIDE']}")
ax_b.set_xscale("log")
ax_b.set_xlabel("bin size (arcmin)")
ax_b.set_ylabel(r"binned - per-galaxy $\mathcal{S}_E$")
ax_b.set_title(f"Binning error against bin size (Rs = {CONFIG['RS_ARCMIN']:.0f}')")
ax_r2 = ax_b.twiny()
ax_r2.set_xscale("log")
ax_r2.set_xlim(np.array(ax_b.get_xlim()) / CONFIG["RS_ARCMIN"])
ax_r2.set_xlabel("bin size / Rs")
ax_b.legend(fontsize=7, ncol=2, framealpha=1)
fig_b.tight_layout()
save_plot(fig_b, f"{CONFIG['TRACT']}_bin_size_test")